<div align="center">

# Data Projects and Hackathon 3  
## Project 
Sergio Fernandez, Alessandro Mecchia 

</div>

In [ ]:
# Standard Library
import gc
import json
import math
import os
import re
import subprocess
import sys
import time
import unicodedata
import warnings
from collections import defaultdict
from pathlib import Path

# Data Processing & Math
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.json as paj
import pyarrow.parquet as pq
import duckdb

# GPU Acceleration (RAPIDS)
import cudf
import cugraph

# Machine Learning & NLP
import lightgbm as lgb
import torch
import yake
from langdetect import DetectorFactory, detect
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline

from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import StandardScaler


# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go


# Interactive Notebook Tools
import ipywidgets as widgets
from IPython.display import display

# HTTP / Networking
import requests

# Configuration

warnings.filterwarnings("ignore")
DetectorFactory.seed = 0  # Ensure consistent language detection

In [ ]:
if torch.cuda.is_available():
    print(f"GPU found: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("GPU not detected. PyTorch is running on CPU.")

In [ ]:
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 5)


## Functions 

In [ ]:
def is_empty(x):
    if x is None:
        return True
    if isinstance(x, str):
        return x.strip() == ""
    if isinstance(x, (list, tuple, set, dict)):
        return len(x) == 0
    if isinstance(x, np.ndarray):
        return x.size == 0
    return pd.isna(x)

In [ ]:
def to_python_nested(x):
    if x is None:
        return None
    if isinstance(x, float) and pd.isna(x):
        return None
    if isinstance(x, np.ndarray):
        return [to_python_nested(v) for v in x.tolist()]
    if isinstance(x, tuple):
        return [to_python_nested(v) for v in x]
    if isinstance(x, list):
        return [to_python_nested(v) for v in x]
    if isinstance(x, dict):
        return {k: to_python_nested(v) for k, v in x.items()}
    return x

def normalize_author_value(v):
    if isinstance(v, np.generic):
        v = v.item()
    if isinstance(v, float) and pd.isna(v):
        return None
    return v

def normalize_authors_cell(x):
    if x is None:
        return None
    if isinstance(x, float) and pd.isna(x):
        return None

    if isinstance(x, np.ndarray):
        x = x.tolist()
    elif isinstance(x, dict):
        x = [x]
    elif isinstance(x, tuple):
        x = list(x)
    elif not isinstance(x, list):
        return None

    out = []
    for item in x:
        if isinstance(item, np.ndarray):
            item = item.tolist()

        if not isinstance(item, dict):
            continue

        clean_item = {k: normalize_author_value(v) for k, v in item.items()}
        out.append(clean_item)

    return out if out else None

def is_empty(x):
    if x is None:
        return True
    if isinstance(x, str):
        return x.strip() == ""
    if isinstance(x, (list, tuple, set, dict)):
        return len(x) == 0
    if isinstance(x, np.ndarray):
        return x.size == 0
    return pd.isna(x)

def valid_authors_sequence(authors):
    if authors is None:
        return False
    if isinstance(authors, float) and pd.isna(authors):
        return False
    try:
        if isinstance(authors, np.ndarray):
            return authors.ndim > 0 and len(authors) > 0
        return isinstance(authors, (list, tuple)) and len(authors) > 0
    except Exception:
        return False
    
def authors_complete(authors):
    if not valid_authors_sequence(authors):
        return False

    for author in authors:
        if not isinstance(author, dict):
            return False
        if is_empty(author.get("id")) or is_empty(author.get("org")):
            return False

    return True

In [ ]:
def check_missing_column(pf, col):
    missing_count = 0
    filled_count = 0
    n = 0

    for rg in range(pf.num_row_groups):
        chunk = pf.read_row_group(rg, columns=[col]).to_pandas()
        missing_mask = chunk[col].apply(is_empty)

        missing_count += missing_mask.sum()
        filled_count += (~missing_mask).sum()
        n += len(chunk)

    print(f"{col} summary (sample of {n:,} rows):\n")
    print(f"missing : {missing_count:,}")
    print(f"filled  : {filled_count:,}")
    print(f"missing %: {missing_count / n * 100:.4f}")

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

OPENALEX_REFERENCES_CACHE = Path("data/cache_openalex_references.json")
MAILTO = ""  # put your email here for polite pool

def load_cache(path):
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_cache(path, cache):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False)

def normalize_doi(x):
    if x is None:
        return ""
    x = str(x).strip().lower()
    return x.replace("https://doi.org/", "").replace("http://doi.org/", "").rstrip("/")

def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]

def fetch_openalex_references_batch(dois_batch, timeout=30):
    session = requests.Session()
    doi_filter = "|".join(dois_batch)

    r = session.get(
        "https://api.openalex.org/works",
        params={
            "filter": f"doi:{doi_filter}",
            "select": "doi,referenced_works",
            "per_page": len(dois_batch),
            "mailto": MAILTO,
        },
        headers={"User-Agent": "DBLP-imputation/1.0"},
        timeout=timeout,
    )
    r.raise_for_status()

    out = {}
    for work in r.json().get("results", []):
        doi = normalize_doi(work.get("doi"))
        refs = work.get("referenced_works", [])
        out[doi] = refs if refs else None
    return out

def fetch_openalex_references_parallel(dois, batch_size=50, max_workers=4, pause_s=0.05):
    results = {}
    batches = list(chunked(dois, batch_size))

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {
            ex.submit(fetch_openalex_references_batch, batch): batch
            for batch in batches
        }

        for i, fut in enumerate(as_completed(futures), start=1):
            batch = futures[fut]
            try:
                batch_result = fut.result()
                results.update(batch_result)
            except Exception as e:
                print(f"Batch failed ({len(batch)} dois): {e}")

            if i % 10 == 0 or i == len(batches):
                print(f"  batches completed: {i}/{len(batches)}")
                if pause_s > 0:
                    time.sleep(pause_s)

    return results

def impute_references_from_openalex_batch(df, batch_size=50, max_workers=4):
    cache = load_cache(OPENALEX_REFERENCES_CACHE)

    mask = df["references"].apply(is_empty) & ~df["doi"].apply(is_empty)
    target_idx = df[mask].index.tolist()
    print(f"references: rows to inspect = {len(target_idx)}")

    dois = [normalize_doi(df.at[idx, "doi"]) for idx in target_idx]
    dois = [d for d in dois if d]

    unique_missing = sorted({d for d in dois if d not in cache})
    print(f"references: DOI not in cache = {len(unique_missing)}")

    if unique_missing:
        fetched = fetch_openalex_references_parallel(
            unique_missing,
            batch_size=batch_size,
            max_workers=max_workers,
        )
        cache.update(fetched)
        save_cache(OPENALEX_REFERENCES_CACHE, cache)

    recovered = 0
    for idx in target_idx:
        doi = normalize_doi(df.at[idx, "doi"])
        refs = cache.get(doi)

        if refs and is_empty(df.at[idx, "references"]):
            df.at[idx, "references"] = refs
            recovered += 1

    print("\nImputation completed for references")
    print(f"Recovered: {recovered}/{len(target_idx)}")
    print(f"Still missing: {df['references'].apply(is_empty).sum()}")

In [ ]:
OPENALEX_ABSTRACT_CACHE = Path("data/cache_openalex_abstracts.json")
MAILTO = "" # put your email here for polite pool

def reconstruct_abstract(inverted_index):
    if not inverted_index:
        return None

    max_pos = max((max(pos) for pos in inverted_index.values() if pos), default=-1)
    if max_pos < 0:
        return None

    words = [""] * (max_pos + 1)
    for word, positions in inverted_index.items():
        for pos in positions:
            words[pos] = word

    text = " ".join(words).strip()
    return text if text else None

def fetch_openalex_abstracts_batch(dois_batch, timeout=30):
    session = requests.Session()
    doi_filter = "|".join(dois_batch)

    r = session.get(
        "https://api.openalex.org/works",
        params={
            "filter": f"doi:{doi_filter}",
            "select": "doi,abstract_inverted_index",
            "per_page": len(dois_batch),
            "mailto": MAILTO,
        },
        headers={"User-Agent": "DBLP-imputation/1.0"},
        timeout=timeout,
    )
    r.raise_for_status()

    out = {}
    for work in r.json().get("results", []):
        doi = normalize_doi(work.get("doi"))
        abstract = reconstruct_abstract(work.get("abstract_inverted_index"))
        out[doi] = abstract
    return out

def fetch_openalex_abstracts_parallel(dois, batch_size=50, max_workers=4, pause_s=0.05):
    results = {}
    batches = list(chunked(dois, batch_size))

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {
            ex.submit(fetch_openalex_abstracts_batch, batch): batch
            for batch in batches
        }

        for i, fut in enumerate(as_completed(futures), start=1):
            batch = futures[fut]
            try:
                batch_result = fut.result()
                results.update(batch_result)
            except Exception as e:
                print(f"Batch failed ({len(batch)} dois): {e}")

            if i % 10 == 0 or i == len(batches):
                print(f"  batches completed: {i}/{len(batches)}")
                if pause_s > 0:
                    time.sleep(pause_s)

    return results

def impute_abstract_from_openalex_batch(df, batch_size=50, max_workers=4):
    cache = load_cache(OPENALEX_ABSTRACT_CACHE)

    mask = df["abstract"].apply(is_empty) & ~df["doi"].apply(is_empty)
    target_idx = df[mask].index.tolist()
    print(f"abstract: rows to inspect = {len(target_idx)}")

    dois = [normalize_doi(df.at[idx, "doi"]) for idx in target_idx]
    dois = [d for d in dois if d]

    unique_missing = sorted({d for d in dois if d not in cache})
    print(f"abstract: DOI not in cache = {len(unique_missing)}")

    if unique_missing:
        fetched = fetch_openalex_abstracts_parallel(
            unique_missing,
            batch_size=batch_size,
            max_workers=max_workers,
        )
        cache.update(fetched)
        save_cache(OPENALEX_ABSTRACT_CACHE, cache)

    recovered = 0
    for idx in target_idx:
        doi = normalize_doi(df.at[idx, "doi"])
        abstract = cache.get(doi)

        if abstract and is_empty(df.at[idx, "abstract"]):
            df.at[idx, "abstract"] = abstract
            recovered += 1

    print("\nImputation completed for abstract")
    print(f"Recovered: {recovered}/{len(target_idx)}")
    print(f"Still missing: {df['abstract'].apply(is_empty).sum()}")


In [ ]:
OPENALEX_VENUE_CACHE = Path("data/cache_openalex_venue.json")
MAILTO = "" # put your email here for polite pool

def extract_venue_from_work(data):
    if not isinstance(data, dict):
        return None
    venue = data.get("primary_location", {}).get("raw_source_name")
    return venue if venue else None

def fetch_openalex_venue_batch(dois_batch, timeout=30):
    session = requests.Session()
    doi_filter = "|".join(dois_batch)

    r = session.get(
        "https://api.openalex.org/works",
        params={
            "filter": f"doi:{doi_filter}",
            "select": "doi,primary_location",
            "per_page": len(dois_batch),
            "mailto": MAILTO,
        },
        headers={"User-Agent": "DBLP-imputation/1.0"},
        timeout=timeout,
    )
    r.raise_for_status()

    out = {}
    for work in r.json().get("results", []):
        doi = normalize_doi(work.get("doi"))
        venue = extract_venue_from_work(work)
        out[doi] = venue
    return out

def fetch_openalex_venue_parallel(dois, batch_size=50, max_workers=4, pause_s=0.05):
    results = {}
    batches = list(chunked(dois, batch_size))

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {
            ex.submit(fetch_openalex_venue_batch, batch): batch
            for batch in batches
        }

        for i, fut in enumerate(as_completed(futures), start=1):
            batch = futures[fut]
            try:
                batch_result = fut.result()
                results.update(batch_result)
            except Exception as e:
                print(f"Batch failed ({len(batch)} dois): {e}")

            if i % 10 == 0 or i == len(batches):
                print(f"  batches completed: {i}/{len(batches)}")
                if pause_s > 0:
                    time.sleep(pause_s)

    return results

def impute_venue_from_openalex_batch(df, batch_size=50, max_workers=4):
    cache = load_cache(OPENALEX_VENUE_CACHE)

    mask = df["venue"].apply(is_empty) & ~df["doi"].apply(is_empty)
    target_idx = df[mask].index.tolist()
    print(f"venue: rows to inspect = {len(target_idx)}")

    dois = [normalize_doi(df.at[idx, "doi"]) for idx in target_idx]
    dois = [d for d in dois if d]

    unique_missing = sorted({d for d in dois if d not in cache})
    print(f"venue: DOI not in cache = {len(unique_missing)}")

    if unique_missing:
        fetched = fetch_openalex_venue_parallel(
            unique_missing,
            batch_size=batch_size,
            max_workers=max_workers,
        )
        cache.update(fetched)
        save_cache(OPENALEX_VENUE_CACHE, cache)

    recovered = 0
    for idx in target_idx:
        doi = normalize_doi(df.at[idx, "doi"])
        venue = cache.get(doi)

        if venue and is_empty(df.at[idx, "venue"]):
            df.at[idx, "venue"] = venue
            recovered += 1

    print("\nImputation completed for venue")
    print(f"Recovered: {recovered}/{len(target_idx)}")
    print(f"Still missing: {df['venue'].apply(is_empty).sum()}")


In [ ]:
OPENALEX_KEYWORDS_CACHE = Path("data/cache_openalex_keywords.json")
MAILTO = "" # put your email here for polite pool

def extract_keywords_from_work(data):
    if not isinstance(data, dict):
        return None

    concepts = data.get("concepts", [])
    keywords = [
        c["display_name"]
        for c in concepts
        if c.get("level", 0) >= 1 and c.get("score", 0) >= 0.3
    ]
    return keywords if keywords else None

def fetch_openalex_keywords_batch(dois_batch, timeout=30):
    session = requests.Session()
    doi_filter = "|".join(dois_batch)

    r = session.get(
        "https://api.openalex.org/works",
        params={
            "filter": f"doi:{doi_filter}",
            "select": "doi,concepts",
            "per_page": len(dois_batch),
            "mailto": MAILTO,
        },
        headers={"User-Agent": "DBLP-imputation/1.0"},
        timeout=timeout,
    )
    r.raise_for_status()

    out = {}
    for work in r.json().get("results", []):
        doi = normalize_doi(work.get("doi"))
        keywords = extract_keywords_from_work(work)
        out[doi] = keywords
    return out

def fetch_openalex_keywords_parallel(dois, batch_size=50, max_workers=4, pause_s=0.05):
    results = {}
    batches = list(chunked(dois, batch_size))

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {
            ex.submit(fetch_openalex_keywords_batch, batch): batch
            for batch in batches
        }

        for i, fut in enumerate(as_completed(futures), start=1):
            batch = futures[fut]
            try:
                batch_result = fut.result()
                results.update(batch_result)
            except Exception as e:
                print(f"Batch failed ({len(batch)} dois): {e}")

            if i % 10 == 0 or i == len(batches):
                print(f"  batches completed: {i}/{len(batches)}")
                if pause_s > 0:
                    time.sleep(pause_s)

    return results

def impute_keywords_from_openalex_batch(df, batch_size=50, max_workers=4):
    cache = load_cache(OPENALEX_KEYWORDS_CACHE)

    mask = df["keywords"].apply(is_empty) & ~df["doi"].apply(is_empty)
    target_idx = df[mask].index.tolist()
    print(f"keywords: rows to inspect = {len(target_idx)}")

    dois = [normalize_doi(df.at[idx, "doi"]) for idx in target_idx]
    dois = [d for d in dois if d]

    unique_missing = sorted({d for d in dois if d not in cache})
    print(f"keywords: DOI not in cache = {len(unique_missing)}")

    if unique_missing:
        fetched = fetch_openalex_keywords_parallel(
            unique_missing,
            batch_size=batch_size,
            max_workers=max_workers,
        )
        cache.update(fetched)
        save_cache(OPENALEX_KEYWORDS_CACHE, cache)

    recovered = 0
    for idx in target_idx:
        doi = normalize_doi(df.at[idx, "doi"])
        keywords = cache.get(doi)

        if keywords and is_empty(df.at[idx, "keywords"]):
            df.at[idx, "keywords"] = keywords
            recovered += 1

    print("\nImputation completed for keywords")
    print(f"Recovered: {recovered}/{len(target_idx)}")
    print(f"Still missing: {df['keywords'].apply(is_empty).sum()}")


In [ ]:
OPENALEX_DOC_TYPE_CACHE = Path("data/cache_openalex_doc_type.json")

def fetch_openalex_doc_type_batch(dois_batch, timeout=30):
    session = requests.Session()
    doi_filter = "|".join(dois_batch)

    r = session.get(
        "https://api.openalex.org/works",
        params={
            "filter": f"doi:{doi_filter}",
            "select": "doi,type",
            "per_page": len(dois_batch),
            "mailto": MAILTO,
        },
        headers={"User-Agent": "DBLP-imputation/1.0"},
        timeout=timeout,
    )
    r.raise_for_status()

    out = {}
    for work in r.json().get("results", []):
        doi = normalize_doi(work.get("doi"))
        doc_type = work.get("type")
        out[doi] = doc_type if doc_type else None
    return out

def fetch_openalex_doc_type_parallel(dois, batch_size=50, max_workers=4, pause_s=0.05):
    results = {}
    batches = list(chunked(dois, batch_size))

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {
            ex.submit(fetch_openalex_doc_type_batch, batch): batch
            for batch in batches
        }

        for i, fut in enumerate(as_completed(futures), start=1):
            batch = futures[fut]
            try:
                batch_result = fut.result()
                results.update(batch_result)
            except Exception as e:
                print(f"Batch failed ({len(batch)} dois): {e}")

            if i % 10 == 0 or i == len(batches):
                print(f"  batches completed: {i}/{len(batches)}")
                if pause_s > 0:
                    time.sleep(pause_s)

    return results

def impute_doc_type_from_openalex_batch(df, batch_size=50, max_workers=4):
    cache = load_cache(OPENALEX_DOC_TYPE_CACHE)

    mask = df["doc_type"].apply(is_empty) & ~df["doi"].apply(is_empty)
    target_idx = df[mask].index.tolist()
    print(f"doc_type: rows to inspect = {len(target_idx)}")

    dois = [normalize_doi(df.at[idx, "doi"]) for idx in target_idx]
    dois = [d for d in dois if d]

    unique_missing = sorted({d for d in dois if d not in cache})
    print(f"doc_type: DOI not in cache = {len(unique_missing)}")

    if unique_missing:
        fetched = fetch_openalex_doc_type_parallel(
            unique_missing,
            batch_size=batch_size,
            max_workers=max_workers,
        )
        cache.update(fetched)
        save_cache(OPENALEX_DOC_TYPE_CACHE, cache)

    recovered = 0
    for idx in target_idx:
        doi = normalize_doi(df.at[idx, "doi"])
        doc_type = cache.get(doi)

        if doc_type and is_empty(df.at[idx, "doc_type"]):
            df.at[idx, "doc_type"] = doc_type
            recovered += 1

    print("\nImputation completed for doc_type")
    print(f"Recovered: {recovered}/{len(target_idx)}")
    print(f"Still missing: {df['doc_type'].apply(is_empty).sum()}")


In [ ]:
import math
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from collections import defaultdict
from langdetect import detect, DetectorFactory
import yake

DetectorFactory.seed = 0

SUPPORTED_LANGS = {"en", "de", "fr", "pt", "es", "it", "zh", "ja", "ko", "ru"}

YAKE_LANG_MAP = {
    "en": "en",
    "it": "it",
    "fr": "fr",
    "de": "de",
    "es": "es",
    "pt": "pt",
    "zh": "zh",
    "ja": "ja",
    "ko": "ko",
    "ru": "ru",
}


In [ ]:
def detect_lang(row):
    parts = []
    for col in ["title", "abstract", "venue"]:
        val = row.get(col)
        if isinstance(val, str) and val.strip():
            parts.append(val.strip())

    text = " ".join(parts)

    if len(text) < 30:
        return None

    try:
        pred = detect(text)
        return pred if pred in SUPPORTED_LANGS else None
    except Exception:
        return None


In [ ]:
from collections import defaultdict

def valid_authors_sequence(authors):
    if authors is None:
        return False
    if isinstance(authors, float) and pd.isna(authors):
        return False

    try:
        if isinstance(authors, np.ndarray):
            return authors.ndim > 0 and len(authors) > 0
        return isinstance(authors, (list, tuple)) and len(authors) > 0
    except Exception:
        return False


def build_author_mappings(input_path):
    pf = pq.ParquetFile(input_path)

    name_to_id = {}
    name_to_year_orgs = defaultdict(list)

    for rg in range(pf.num_row_groups):
        print(f"[authors mapping] Row group {rg + 1}/{pf.num_row_groups}")

        chunk = pf.read_row_group(rg, columns=["authors", "year"]).to_pandas()

        for _, row in chunk.iterrows():
            year = row.get("year")
            authors = row.get("authors")

            if not valid_authors_sequence(authors):
                continue

            parsed_year = None
            try:
                if year is not None and not (isinstance(year, float) and pd.isna(year)):
                    parsed_year = int(year)
            except Exception:
                parsed_year = None

            for author in authors:
                if not isinstance(author, dict):
                    continue

                name = author.get("name")
                author_id = author.get("id")
                org = author.get("org")

                if isinstance(name, str):
                    name = name.strip()

                if not name:
                    continue

                if isinstance(author_id, str) and author_id.strip() and name not in name_to_id:
                    name_to_id[name] = author_id.strip()

                if isinstance(org, str) and org.strip() and parsed_year is not None:
                    name_to_year_orgs[name].append((parsed_year, org.strip()))

    print(f"Created mapping for {len(name_to_id):,} unique author names")
    print(f"Created org history for {len(name_to_year_orgs):,} author names")

    return name_to_id, name_to_year_orgs


In [ ]:
def impute_authors_id_chunk(chunk, name_to_id):
    recovered = 0

    for idx, row in chunk.iterrows():
        authors = row.get("authors")

        if not valid_authors_sequence(authors):
            continue

        changed = False
        for author in authors:
            if not isinstance(author, dict):
                continue

            name = author.get("name")
            author_id = author.get("id")

            if isinstance(name, str):
                name = name.strip()

            if not author_id and name and name in name_to_id:
                author["id"] = name_to_id[name]
                recovered += 1
                changed = True

        if changed:
            chunk.at[idx, "authors"] = authors

    print(f"Recovered {recovered:,} author IDs")
    return chunk


In [ ]:
def impute_authors_org_chunk(chunk, name_to_year_orgs, max_year_diff=1):
    recovered = 0

    for idx, row in chunk.iterrows():
        year = row.get("year")
        authors = row.get("authors")

        if not valid_authors_sequence(authors):
            continue

        try:
            if year is None or (isinstance(year, float) and pd.isna(year)):
                continue
            year = int(year)
        except Exception:
            continue

        changed = False
        for author in authors:
            if not isinstance(author, dict):
                continue

            name = author.get("name")
            org = author.get("org")

            if isinstance(name, str):
                name = name.strip()

            if org or not name:
                continue

            candidates = name_to_year_orgs.get(name)
            if not candidates:
                continue

            best_org = None
            best_distance = float("inf")

            for cand_year, cand_org in candidates:
                distance = abs(cand_year - year)
                if distance <= max_year_diff and distance < best_distance:
                    best_org = cand_org
                    best_distance = distance
                    if distance == 0:
                        break

            if best_org:
                author["org"] = best_org
                recovered += 1
                changed = True

        if changed:
            chunk.at[idx, "authors"] = authors

    print(f"Recovered {recovered:,} author ORGs")
    return chunk


## 1. Data Creation

###  1.2. Parquet Creation 

da eseguire una sola volta 

In [ ]:
DATA_DIR = Path("data")
SOURCE_PATH = DATA_DIR / "DBLP-Citation-network-V18.jsonl"
TARGET_PATH = DATA_DIR / "DBLP-Citation-network-V18.parquet"
BLOCK_SIZE = 64 * 1024 * 1024  

if not SOURCE_PATH.exists():
    raise FileNotFoundError(f"File non trovato: {SOURCE_PATH}")

if pa.Codec.is_available("zstd"):
    COMPRESSION = "zstd"
elif pa.Codec.is_available("snappy"):
    COMPRESSION = "snappy"
else:
    COMPRESSION = None

print(f"Input : {SOURCE_PATH} ({SOURCE_PATH.stat().st_size / 1024**3:.2f} GiB)")
print(f"Output: {TARGET_PATH}")
print(f"Compressione: {COMPRESSION}")
print(f"Block size: {BLOCK_SIZE / 1024**2:.0f} MiB")

In [ ]:
DROP_COLS = {"page_start", "page_end", "volume", "issue", "issn", "isbn", "url"}

if TARGET_PATH.exists():
    print(f"Parquet already exists, skipping conversion.")
else:
    reader = paj.open_json(
        SOURCE_PATH,
        read_options=paj.ReadOptions(block_size=BLOCK_SIZE),
    )

    writer = None
    rows_written = 0
    batches_written = 0
    started_at = time.perf_counter()

    try:
        while True:
            try:
                batch = reader.read_next_batch()
            except StopIteration:
                break

            keep_cols = [name for name in batch.schema.names if name not in DROP_COLS]
            batch = batch.select(keep_cols)

            if writer is None:
                writer = pq.ParquetWriter(
                    TARGET_PATH,
                    batch.schema,
                    compression=COMPRESSION,
                )

            writer.write_batch(batch)
            rows_written += batch.num_rows
            batches_written += 1

            if batches_written % 25 == 0:
                elapsed = time.perf_counter() - started_at
                print(f"Batch: {batches_written:>5} | Rows: {rows_written:>12,} | Elapsed: {elapsed:>8.1f}s")

        if writer is None:
            raise RuntimeError("JSONL file seems empty: no batch read.")
    finally:
        reader.close()
        if writer is not None:
            writer.close()

    elapsed = time.perf_counter() - started_at
    print(f"Conversion completed in {elapsed:.1f}s")
    print(f"Rows written : {rows_written:,}")
    print(f"JSONL size   : {SOURCE_PATH.stat().st_size / 1024**3:.2f} GiB")
    print(f"Parquet size : {TARGET_PATH.stat().st_size / 1024**3:.2f} GiB")

### 1.2 Data Creation with DuckDB

Prima di lavorare su batch o sample, facciamo profiling e data quality sull'intero dataset direttamente sul parquet via DuckDB. In questo modo i conteggi dei missing, le distribuzioni per anno e la selezione dei candidati per imputazione sono globali e non dipendono dal sample da 10k record.

In [ ]:
PARQUET_PATH = Path("data/DBLP-Citation-network-V18.parquet")


con = duckdb.connect(":memory:")

# con = duckdb.connect(r"C:\Users\ms\Desktop\dblp.duckdb")

con.execute("PRAGMA threads=4;")
con.execute("PRAGMA enable_progress_bar;")

parquet_str = PARQUET_PATH.as_posix()

con.execute(f"""
    CREATE OR REPLACE VIEW papers AS
    SELECT *
    FROM read_parquet('{parquet_str}');
""")

In [ ]:
display(con.execute("SELECT * FROM papers LIMIT 5").fetchdf())

In [ ]:
# PROFILE_DB_PATH = DATA_DIR / "dblp_profiling.duckdb"

# con_full = duckdb.connect(PROFILE_DB_PATH.as_posix())
# con_full.execute("PRAGMA threads=4;")
# con_full.execute("PRAGMA enable_progress_bar;")

# con_full.execute(f"""
#     CREATE OR REPLACE VIEW papers_full AS
#     SELECT *
#     FROM read_parquet('{parquet_str}');
# """)

# total_rows_full = con_full.execute("SELECT COUNT(*) FROM papers_full").fetchone()[0]
# print(f"Profiling DB: {PROFILE_DB_PATH}")
# print(f"Total rows available in DuckDB: {total_rows_full:,}")

## 2. Data Exploration  

In [ ]:
# pf = pq.ParquetFile(r"data/DBLP-Citation-network-V18.parquet")

# print(f"Total rows   : {pf.metadata.num_rows:,}")
# print(f"Columns      : {pf.metadata.num_columns}")
# print(f"Row groups   : {pf.metadata.num_row_groups}")

In [ ]:
total_rows = con.execute("""
    SELECT COUNT(*)
    FROM papers
""").fetchone()[0]

n_columns = con.execute("""
    DESCRIBE SELECT * FROM papers
""").fetchdf().shape[0]

row_groups = con.execute(f"""
    SELECT COUNT(DISTINCT row_group_id)
    FROM parquet_metadata('{parquet_str}')
""").fetchone()[0]

print(f"Total rows   : {total_rows:,}")
print(f"Columns      : {n_columns}")
print(f"Row groups   : {row_groups}")


In [ ]:
schema_df = con.execute("""
    DESCRIBE SELECT * FROM papers
""").fetchdf()

for i, name in enumerate(schema_df["column_name"]):
    print(f"{i}. {name}")


In [ ]:
schema_df = con.execute("""
    DESCRIBE SELECT * FROM papers
""").fetchdf()

columns = schema_df["column_name"].tolist()

dropdown = widgets.Dropdown(options=columns, description="Colonna:")
output = widgets.Output()

def on_change(change):
    if change["type"] == "change" and change["name"] == "value":
        col = change["new"]
        with output:
            output.clear_output()
            df = con.execute(f'''
                SELECT "{col}"
                FROM papers
                LIMIT 10
            ''').fetchdf()
            display(df)

dropdown.observe(on_change)

display(dropdown, output)

In [ ]:
paper = con.execute("""
    SELECT *
    FROM papers
    LIMIT 1
""").fetchdf().iloc[0]

for col, val in paper.items():
    print(f"{col:15}: {val}")

### 2.1. Data Selection

In [ ]:
con.execute("""
    CREATE OR REPLACE VIEW papers_profile_base AS
    SELECT
        *,
        CASE WHEN title IS NULL OR TRIM(CAST(title AS VARCHAR)) = '' THEN TRUE ELSE FALSE END AS flag_missing_title,
        CASE WHEN abstract IS NULL OR TRIM(CAST(abstract AS VARCHAR)) = '' THEN TRUE ELSE FALSE END AS flag_missing_abstract,
        CASE WHEN doi IS NULL OR TRIM(CAST(doi AS VARCHAR)) = '' THEN TRUE ELSE FALSE END AS flag_missing_doi,
        CASE WHEN lang IS NULL OR TRIM(CAST(lang AS VARCHAR)) = '' THEN TRUE ELSE FALSE END AS flag_missing_lang,
        CASE WHEN venue IS NULL OR TRIM(CAST(venue AS VARCHAR)) = '' THEN TRUE ELSE FALSE END AS flag_missing_venue,
        CASE WHEN doc_type IS NULL OR TRIM(CAST(doc_type AS VARCHAR)) = '' THEN TRUE ELSE FALSE END AS flag_missing_doc_type,
        CASE WHEN COALESCE(array_length(keywords), 0) = 0 THEN TRUE ELSE FALSE END AS flag_missing_keywords,
        CASE WHEN COALESCE(array_length("references"), 0) = 0 THEN TRUE ELSE FALSE END AS flag_missing_references,
        CASE WHEN COALESCE(array_length(authors), 0) = 0 THEN TRUE ELSE FALSE END AS flag_missing_authors,
        CASE WHEN TRY_CAST(year AS INTEGER) IS NULL THEN TRUE ELSE FALSE END AS flag_year_missing,
        TRY_CAST(year AS INTEGER) AS year_int

    FROM papers
""")


### 2.1.1. Missing Values

In [ ]:
missing_summary = con.execute("""
    SELECT * FROM (
        SELECT 'title' AS feature, SUM(CAST(flag_missing_title AS INTEGER)) AS count_rows FROM papers_profile_base
        UNION ALL
        SELECT 'abstract', SUM(CAST(flag_missing_abstract AS INTEGER)) FROM papers_profile_base
        UNION ALL
        SELECT 'doi', SUM(CAST(flag_missing_doi AS INTEGER)) FROM papers_profile_base
        UNION ALL
        SELECT 'lang', SUM(CAST(flag_missing_lang AS INTEGER)) FROM papers_profile_base
        UNION ALL
        SELECT 'venue', SUM(CAST(flag_missing_venue AS INTEGER)) FROM papers_profile_base
        UNION ALL
        SELECT 'doc_type', SUM(CAST(flag_missing_doc_type AS INTEGER)) FROM papers_profile_base
        UNION ALL
        SELECT 'keywords', SUM(CAST(flag_missing_keywords AS INTEGER)) FROM papers_profile_base
        UNION ALL
        SELECT 'references', SUM(CAST(flag_missing_references AS INTEGER)) FROM papers_profile_base
        UNION ALL
        SELECT 'authors', SUM(CAST(flag_missing_authors AS INTEGER)) FROM papers_profile_base
        UNION ALL
        SELECT 'year', SUM(CAST(flag_year_missing AS INTEGER)) FROM papers_profile_base
    )
    ORDER BY count_rows DESC
""").fetchdf()

total_rows = con.execute("SELECT COUNT(*) FROM papers_profile_base").fetchone()[0]
missing_summary["pct_rows"] = missing_summary["count_rows"] / total_rows * 100
missing_summary

In [ ]:
plt.figure(figsize=(14, 7))
ax = sns.barplot(data=missing_summary, x="feature", y="pct_rows", color="#C44E52", width=0.7)

plt.title("Missing values by feature", fontsize=16, fontweight='bold', pad=20)
plt.xlabel("Feature", fontsize=13, fontweight='bold')
plt.ylabel("Missing (%)", fontsize=13, fontweight='bold')

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=5, fontsize=11, fontweight='bold')

plt.xticks(rotation=45, ha='right', fontsize=11)
plt.yticks(fontsize=11)

ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.7)
ax.set_axisbelow(True)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCCCCC')
ax.spines['bottom'].set_color('#CCCCCC')

plt.tight_layout()
plt.show()

In [ ]:
authors_missing_summary = con.execute("""
WITH authors_exploded AS (
    SELECT
        p.id AS paper_id,
        a.author AS author
    FROM papers_profile_base p
    CROSS JOIN UNNEST(p.authors) AS a(author)
)
SELECT
    COUNT(*) AS total_author_rows,
    SUM(CASE WHEN author.id IS NULL OR TRIM(CAST(author.id AS VARCHAR)) = '' THEN 1 ELSE 0 END) AS missing_author_id,
    SUM(CASE WHEN author.name IS NULL OR TRIM(CAST(author.name AS VARCHAR)) = '' THEN 1 ELSE 0 END) AS missing_author_name,
    SUM(CASE WHEN author.org IS NULL OR TRIM(CAST(author.org AS VARCHAR)) = '' THEN 1 ELSE 0 END) AS missing_author_org
FROM authors_exploded
""").fetchdf()

display(authors_missing_summary)

In [ ]:
missing_by_year = con.execute("""
    SELECT
        year_int AS year,
        COUNT(*) AS n_papers,
        SUM(CAST(flag_missing_abstract AS INTEGER)) AS missing_abstract,
        SUM(CAST(flag_missing_doi AS INTEGER)) AS missing_doi,
        SUM(CAST(flag_missing_venue AS INTEGER)) AS missing_venue,
        SUM(CAST(flag_missing_doc_type AS INTEGER)) AS missing_doc_type,
        SUM(CAST(flag_missing_keywords AS INTEGER)) AS missing_keywords,
        SUM(CAST(flag_missing_references AS INTEGER)) AS missing_references
    FROM papers_profile_base
    WHERE year_int IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").fetchdf()

display(missing_by_year.head(5))

escludiamo gli anni fino a 1990 al 2027 non incluso e siccome vogliamo imputare abstract e reference con doi, droppiamo anche subito i paper che hanno missing DOI e una delle due features.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW papers_scope AS
SELECT *
FROM papers_profile_base
WHERE year_int > 1990
  AND year_int < 2027
  AND NOT (
      flag_missing_doi
      AND (flag_missing_abstract OR flag_missing_references)
  )
""")

In [ ]:
missing_counts_query = con.execute("""
WITH missing_per_paper AS (
    SELECT
        id,
        CAST(flag_missing_title AS INTEGER) +
        CAST(flag_missing_abstract AS INTEGER) +
        CAST(flag_missing_doi AS INTEGER) +
        CAST(flag_missing_lang AS INTEGER) +
        CAST(flag_missing_venue AS INTEGER) +
        CAST(flag_missing_doc_type AS INTEGER) +
        CAST(flag_missing_keywords AS INTEGER) +
        CAST(flag_missing_references AS INTEGER) +
        CAST(flag_missing_authors AS INTEGER) +
        CAST(flag_year_missing AS INTEGER) AS num_missing
    FROM papers_scope
)
SELECT
    num_missing,
    COUNT(*) AS count_papers
FROM missing_per_paper
GROUP BY num_missing
ORDER BY num_missing
""").fetchdf()


In [ ]:
plt.figure(figsize=(14, 7))
ax = sns.barplot(data=missing_counts_query, x="num_missing", y="count_papers", color="#C44E52", width=0.7)

plt.title("Distribution of missing values per paper", fontsize=16, fontweight='bold', pad=20)
plt.xlabel("Number of missing values", fontsize=13, fontweight='bold')
plt.ylabel("Number of papers", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Fix: use count_papers instead of count_rows
missing_counts_query["pct_papers"] = missing_counts_query["count_papers"] / missing_counts_query["count_papers"].sum() * 100

per risparmiare spazio droppiamo subito paper con missing importanti >= 2

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW papers_working AS
SELECT *
FROM (
    SELECT
        *,
        (
            CAST(flag_missing_title AS INTEGER) +
            CAST(flag_missing_abstract AS INTEGER) +
            CAST(flag_missing_lang AS INTEGER) +
            CAST(flag_missing_venue AS INTEGER) +
            CAST(flag_missing_doc_type AS INTEGER) +
            CAST(flag_missing_keywords AS INTEGER) +
            CAST(flag_missing_references AS INTEGER) +
            CAST(flag_missing_authors AS INTEGER) +
            CAST(flag_year_missing AS INTEGER)
        ) AS num_missing
    FROM papers_scope
) t
WHERE num_missing <= 1
""")

In [ ]:
con.execute("""
SELECT
    COUNT(*) AS total_papers,
    SUM(CAST(flag_missing_title AS INTEGER)) AS missing_title,
    SUM(CAST(flag_missing_abstract AS INTEGER)) AS missing_abstract,
    SUM(CAST(flag_missing_doi AS INTEGER)) AS missing_doi,
    SUM(CAST(flag_missing_references AS INTEGER)) AS missing_references,
    SUM(CAST(flag_missing_authors AS INTEGER)) AS missing_authors,
    SUM(CAST(flag_year_missing AS INTEGER)) AS missing_year,
    SUM(CAST(flag_missing_lang AS INTEGER)) AS missing_lang,
    SUM(CAST(flag_missing_venue AS INTEGER)) AS missing_venue,
    SUM(CAST(flag_missing_doc_type AS INTEGER)) AS missing_doc_type
FROM papers_working
""").fetchdf()

In [ ]:
missing_by_year = con.execute("""
    SELECT
        year_int AS year,
        SUM(CAST(flag_missing_abstract AS INTEGER)) AS abstract,
        SUM(CAST(flag_missing_doi AS INTEGER)) AS doi,
        SUM(CAST(flag_missing_venue AS INTEGER)) AS venue,
        SUM(CAST(flag_missing_doc_type AS INTEGER)) AS doc_type,
        SUM(CAST(flag_missing_keywords AS INTEGER)) AS keywords,
        SUM(CAST(flag_missing_references AS INTEGER)) AS "references"
    FROM papers_working
    GROUP BY 1
    ORDER BY 1
""").fetchdf()

missing_by_year = missing_by_year.set_index("year")

missing_by_year.plot(
    kind="bar",
    stacked=True,
    figsize=(16, 7),
    colormap="tab20"
)

plt.title("Missing values by year and feature")
plt.xlabel("Year")
plt.ylabel("Missing count")
plt.tight_layout()
plt.show()

In [ ]:
MAX_OPENALEX_CALLS = 90000

budget_df = con.execute("""
    SELECT
        COUNT(*) FILTER (WHERE flag_missing_abstract) AS n_missing_abstract,
        COUNT(*) FILTER (WHERE flag_missing_references) AS n_missing_references
    FROM papers_working
""").fetchdf()

n_missing_abstract = int(budget_df.loc[0, "n_missing_abstract"])
n_missing_references = int(budget_df.loc[0, "n_missing_references"])
ref_budget = max(0, MAX_OPENALEX_CALLS - n_missing_abstract)

print({
    "missing_abstract_all": n_missing_abstract,
    "missing_references_pool": n_missing_references,
    "reference_sample_budget": ref_budget
})


In [ ]:
# segment_summary = con.execute(f"""
# SELECT
#     paper_segment,
#     COUNT(*) AS n_papers,
#     SUM(CASE WHEN NOT flag_missing_abstract THEN 1 ELSE 0 END) AS has_abstract,
#     SUM(CASE WHEN NOT flag_missing_references THEN 1 ELSE 0 END) AS has_references,
#     SUM(CASE WHEN NOT flag_missing_doi THEN 1 ELSE 0 END) AS has_doi,
#     SUM(CASE WHEN COALESCE(array_length("references"), 0) >= {MIN_REFERENCES_RECOVERABLE} THEN 1 ELSE 0 END) AS refs_ge_threshold
# FROM papers_record_quality_preimputation
# GROUP BY 1
# """).fetchdf()

In [ ]:
ref_year_counts = con.execute("""
    SELECT
        year_int AS year,
        COUNT(*) AS n_year
    FROM papers_working
    WHERE flag_missing_references
      AND NOT flag_missing_abstract
    GROUP BY 1
    ORDER BY 1
""").fetchdf()

if len(ref_year_counts) > 0 and ref_budget > 0:
    ref_year_counts["quota_float"] = (
        ref_year_counts["n_year"] / ref_year_counts["n_year"].sum() * ref_budget
    )
    ref_year_counts["k"] = ref_year_counts["quota_float"].astype(int)

    remaining = ref_budget - int(ref_year_counts["k"].sum())
    if remaining > 0:
        ref_year_counts["frac"] = ref_year_counts["quota_float"] - ref_year_counts["k"]
        extra_idx = ref_year_counts.nlargest(remaining, "frac").index
        ref_year_counts.loc[extra_idx, "k"] += 1
else:
    ref_year_counts["quota_float"] = 0.0
    ref_year_counts["k"] = 0

ref_alloc = ref_year_counts[["year", "k"]].copy()

con.register("ref_alloc_df", ref_alloc)


we keep only good and recoverable

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE papers_final_ids AS
WITH ref_ranked AS (
    SELECT
        p.id,
        p.year_int,
        ROW_NUMBER() OVER (
            PARTITION BY p.year_int
            ORDER BY random()
        ) AS rn
    FROM papers_working p
    WHERE p.flag_missing_references
      AND NOT p.flag_missing_abstract
),
ref_sample AS (
    SELECT r.id
    FROM ref_ranked r
    JOIN ref_alloc_df a
      ON r.year_int = a.year
    WHERE r.rn <= a.k
)
SELECT p.id
FROM papers_working p
LEFT JOIN ref_sample s
  ON p.id = s.id
WHERE
    NOT p.flag_missing_references
    OR s.id IS NOT NULL
""")

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW papers_final_original_filtered AS
SELECT p.*
FROM papers p
JOIN papers_final_ids f
  ON p.id = f.id
""")


crea il parquet finale

In [ ]:
output_path = "data/first_dataset.parquet"

if Path(output_path).exists() or Path("data/final_dataset.parquet").exists() or Path("data/dataset.parquet").exists():
    print(f"already exists: {output_path}")
else:
    con.execute(f"""
    COPY papers_final_original_filtered
    TO '{output_path}'
    (FORMAT PARQUET)
    """)

    print("saved:", output_path)

spiegare che ci interessa imputare solo le variabili che pensiamo possano essere utili per predirre il numero di citazioni o se un paper cita un'altro, ma prima potrebbe essere utile analizzare la distribuzione nei anni dei missing value.

In [ ]:
# pf = pq.ParquetFile(r"data/first_dataset.parquet")

# print(f"Total rows   : {pf.metadata.num_rows:,}")
# print(f"Columns      : {pf.metadata.num_columns}")
# print(f"Row groups   : {pf.metadata.num_row_groups}")

## 3. Dataset Imputation 

### 3.1. Abstract, References, Venue and Doc_Type Imputation

In [ ]:
input_path = "data/first_dataset.parquet"
output_path = "data/middle_dataset.parquet"

if Path(output_path).exists() or Path("data/final_dataset.parquet").exists() or Path("data/dataset.parquet").exists():
    print(f"already exists: {output_path}")
else:
    pf = pq.ParquetFile(input_path)
    writer = None

    try:
        for rg in range(pf.num_row_groups):
            print(f"\n[openalex] Row group {rg + 1}/{pf.num_row_groups}")

            chunk = pf.read_row_group(rg).to_pandas()

            impute_abstract_from_openalex_batch(
                chunk,
                batch_size=50,
                max_workers=4,
            )

            impute_references_from_openalex_batch(
                chunk,
                batch_size=50,
                max_workers=4,
            )

            impute_keywords_from_openalex_batch(
                chunk,
                batch_size=50,
                max_workers=4,
            )

            impute_venue_from_openalex_batch(
                chunk,
                batch_size=50,
                max_workers=4,
            )

            impute_doc_type_from_openalex_batch(
                chunk,
                batch_size=50,
                max_workers=4,
            )

            table = pa.Table.from_pandas(chunk, preserve_index=False)

            if writer is None:
                writer = pq.ParquetWriter(output_path, table.schema)

            writer.write_table(table)

    finally:
        if writer is not None:
            writer.close()

    print(f"saved: {output_path}")

In [ ]:
# clean ram
for var_name in ["pf", "chunk", "table", "writer"]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()

### 3.2. Lang and Authors imputation

In [ ]:
input_path = "data/middle_dataset.parquet"
output_path = "data/final_dataset.parquet"

if Path(output_path).exists() or Path("data/dataset.parquet").exists():
    print(f"already exists: {output_path}")
else:

    name_to_id, name_to_year_orgs = build_author_mappings(input_path)

    pf = pq.ParquetFile(input_path)
    writer = None

    try:
        for rg in range(pf.num_row_groups):
            print(f"\n[local imputation] Row group {rg + 1}/{pf.num_row_groups}")

            chunk = pf.read_row_group(rg).to_pandas()

            missing_lang_mask = chunk["lang"].apply(is_empty)
            chunk.loc[missing_lang_mask, "lang"] = (
                chunk.loc[missing_lang_mask].apply(detect_lang, axis=1)
            )


            chunk = impute_authors_id_chunk(chunk, name_to_id)
            chunk = impute_authors_org_chunk(chunk, name_to_year_orgs, max_year_diff=1)

            for col in ["authors", "keywords", "references", "url"]:
                if col in chunk.columns:
                    chunk[col] = chunk[col].apply(to_python_nested)

            chunk["authors"] = chunk["authors"].apply(normalize_authors_cell)

            table = pa.Table.from_pandas(chunk, preserve_index=False)

            if writer is None:
                writer = pq.ParquetWriter(output_path, table.schema)

            writer.write_table(table)

    finally:
        if writer is not None:
            writer.close()

    print(f"saved: {output_path}")

In [ ]:
# clean ram
import gc

for var_name in ["name_to_id", "name_to_year_orgs", "pf", "chunk", "table", "writer"]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()

### 3.3. Analysis Post imputation

In [ ]:
input_path = "data/final_dataset.parquet"
pf = pq.ParquetFile(input_path)

In [ ]:
from collections import Counter
lang_counter = Counter()
n = 0

for rg in range(pf.num_row_groups):
    chunk = pf.read_row_group(rg, columns=["lang"]).to_pandas()

    for val in chunk["lang"]:
        key = "(null)" if is_empty(val) else val
        lang_counter[key] += 1

    n += len(chunk)

lang_counts = pd.Series(lang_counter).sort_values(ascending=False)

print(f"Unique values of 'lang' (sample of {n:,} rows):\n")
print(lang_counts)

In [ ]:
check_missing_column(pf, "abstract")

In [ ]:
check_missing_column(pf, "references")


In [ ]:
check_missing_column(pf, "keywords")

In [ ]:
check_missing_column(pf, "doc_type")

In [ ]:
check_missing_column(pf, "venue")

In [ ]:
missing_authors_id = 0
missing_authors_org = 0
total_authors = 0
papers_with_no_authors = 0

for rg in range(pf.num_row_groups):
    chunk = pf.read_row_group(rg, columns=["authors"]).to_pandas()

    for authors in chunk["authors"]:
        if not valid_authors_sequence(authors):
            papers_with_no_authors += 1
            continue

        for author in authors:
            if not isinstance(author, dict):
                continue

            total_authors += 1

            if is_empty(author.get("id")):
                missing_authors_id += 1

            if is_empty(author.get("org")):
                missing_authors_org += 1

print(f"Authors summary:\n")
print(f"total authors         : {total_authors:,}")
print(f"missing authors.id    : {missing_authors_id:,}")
print(f"missing authors.org   : {missing_authors_org:,}")
print(f"papers with no authors: {papers_with_no_authors:,}")

if total_authors > 0:
    print(f"missing authors.id %  : {missing_authors_id / total_authors * 100:.4f}")
    print(f"missing authors.org % : {missing_authors_org / total_authors * 100:.4f}")

abbiamo imputato il piu possibile, droppiamo tutti i missing che sono rimasti e lavoriamo con il dataset finale da qui in avanti

In [ ]:
def drop_missing(input_path, output_path, required_cols):
    if Path(output_path).exists():
        print(f"already exists: {output_path}")
        return

    pf = pq.ParquetFile(input_path)
    writer = None
    kept_rows = 0
    dropped_rows = 0

    try:
        for rg in range(pf.num_row_groups):
            print(f"\n[filter] Row group {rg + 1}/{pf.num_row_groups}")

            chunk = pf.read_row_group(rg).to_pandas()

            mask = pd.Series(True, index=chunk.index)

            for col in required_cols:
                mask &= ~chunk[col].apply(is_empty)

            # per authors: tieni solo paper con una lista autori valida e non vuota
            mask &= chunk["authors"].apply(valid_authors_sequence)

            kept_rows += int(mask.sum())
            dropped_rows += int((~mask).sum())

            chunk = chunk.loc[mask].copy()

            if len(chunk) == 0:
                continue

            table = pa.Table.from_pandas(chunk, preserve_index=False)

            if writer is None:
                writer = pq.ParquetWriter(output_path, table.schema)

            writer.write_table(table)

    finally:
        if writer is not None:
            writer.close()

In [ ]:
required_cols = [
    "id",
    "title",
    "abstract",
    "year",
    "lang",
    "keywords",
    "references",
    "doi",
    "venue",
    "doc_type",
]

drop_missing(
    input_path="data/final_dataset.parquet",
    output_path="data/dataset.parquet",
    required_cols=required_cols,
)

## 4. Data Visualizzation 

Now that we have the final imputed dataset, we take a step back before moving to feature engineering and modeling. This section gives us a visual overview of the dataset after all cleaning and imputation steps, to make sure the data looks reasonable and well distributed.

In [ ]:
import tempfile

if Path("data/dataset_visualization.parquet").exists():
    df = pd.read_parquet("data/dataset_visualization.parquet")
else:
    pf = pq.ParquetFile("data/dataset.parquet")
    total_rows = pf.metadata.num_rows
    fraction = 50_000 / total_rows

    tmp_files = []
    for batch in pf.iter_batches(batch_size=100_000):
        df_batch = batch.to_pandas()
        del batch
        n = max(1, int(len(df_batch) * fraction))
        sample = df_batch.sample(n, random_state=42)
        del df_batch
        gc.collect()
        tmp = tempfile.NamedTemporaryFile(suffix=".parquet", delete=False)
        sample.to_parquet(tmp.name)
        tmp_files.append(tmp.name)

    df = pd.concat([pd.read_parquet(f) for f in tmp_files], ignore_index=True)
    for f in tmp_files:
        os.remove(f)
    
    df.to_parquet("data/dataset_visualization.parquet", index=False)

To avoid loading the full dataset into memory, we process it in batches, sample a fraction of each batch, write it to a temporary file, and then concatenate all samples at the end. The temporary files are deleted once the final dataframe is assembled.

### 4.1. Paper Distribution 

In [ ]:
year_counts = (
    df["year"]
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
)

year_counts = year_counts.reindex(
    range(year_counts.index.min(), year_counts.index.max() + 1),
    fill_value=0
)
rolling_avg = year_counts.rolling(window=5, center=True, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(15, 6))
ax.bar(
    year_counts.index,
    year_counts.values,
    width=0.85,
    color="#4C78A8",
    edgecolor="#1F2A44",
    linewidth=0.6,
    alpha=0.9,
    label="Papers per year"
)

peak_year = int(year_counts.idxmax())
peak_value = int(year_counts.max())
ax.scatter([peak_year], [peak_value], color="#E45756", s=45, zorder=3)
ax.annotate(
    f"Peak: {peak_value} papers ({peak_year})",
    xy=(peak_year, peak_value),
    xytext=(peak_year - 12, peak_value + 35),
    arrowprops=dict(arrowstyle="->", color="#444444", lw=1),
    fontsize=10
)

tick_years = list(range(year_counts.index.min(), year_counts.index.max() + 1, 5))
if year_counts.index.max() not in tick_years:
    tick_years.append(year_counts.index.max())

ax.set_xticks(tick_years)
ax.set_xlim(year_counts.index.min() - 1, year_counts.index.max() + 1)
ax.set_title("Distribution of papers by publication year", pad=12)
ax.set_xlabel("Publication year")
ax.set_ylabel("Number of papers")
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.legend(frameon=False)
sns.despine(ax=ax)

plt.tight_layout()
plt.show()

The distribution of papers by year shows a steady growth over time, peaking in 2023. The drop in 2025 is expected since the dataset was collected before the end of that year. Overall the distribution looks healthy and consistent with what we observed before filtering.

In [ ]:
tmp = (
    df.dropna(subset=["year"])
      .assign(year=lambda x: x["year"].astype(int))
      .groupby(["year", "doc_type"])
      .size()
      .unstack(fill_value=0)
)

tmp.plot.area(figsize=(14, 6), alpha=0.8)
plt.title("Document types over time")
plt.xlabel("Year")
plt.ylabel("Number of papers")
plt.tight_layout()
plt.show()


Breaking down the distribution by document type, we can see that the dataset is fairly balanced between conference and journal papers, with book-chapters representing a very small minority. This imbalance suggests that book-chapters could be dropped to keep the dataset cleaner.

In [ ]:
#sns.set_theme(style="whitegrid", context="talk")
#plt.rcParams["figure.figsize"] = (12, 5)

### 4.2. Feauture Visualizzation 

#### 4.2.1. Numerical

In [ ]:
numerical = [col for col in df.select_dtypes(include=[np.number]).columns]

In [ ]:
def plot_distribution(feature_idx):
    feature = numerical[feature_idx]
    fig, ax = plt.subplots(figsize=(12, 6))
    
    data = df[feature].dropna()
    years = data.astype(int).value_counts().sort_index()
    ax.bar(years.index, years.values, color='steelblue', alpha=0.7, edgecolor='black', width=0.8)    
    mean = data.mean()
    median = data.median()
    q1 = data.quantile(0.25)
    q3 = data.quantile(0.75)
    std = data.std()
    min_val = data.min()
    max_val = data.max()
    
    ax.axvline(mean, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean:.2f}')
    ax.axvline(median, color='green', linestyle='--', linewidth=2, label=f'Median: {median:.2f}')
    ax.axvline(q1, color='orange', linestyle=':', linewidth=2, label=f'Q1: {q1:.2f}')
    ax.axvline(q3, color='purple', linestyle=':', linewidth=2, label=f'Q3: {q3:.2f}')
    
    ax.set_title(f'Distribution of {feature}', fontsize=14, fontweight='bold')
    ax.set_xlabel(feature, fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    
    stats_text = f'Min: {min_val:.2f} | Max: {max_val:.2f} | Std: {std:.2f}\n\nMean: {mean:.2f}\nMedian: {median:.2f}\nQ1: {q1:.2f}\nQ3: {q3:.2f}'
    ax.text(0.99, 0.97, stats_text, transform=ax.transAxes, fontsize=10, 
            verticalalignment='top', horizontalalignment='right', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)
    
    plt.tight_layout()
    plt.show()

feature_slider = widgets.IntSlider(min=0, max=len(numerical)-1, step=1, value=0)

widgets.interactive(plot_distribution, feature_idx=feature_slider)

#### 4.2.2. Categorical

In [ ]:
categorical = [col for col in df.columns if df[col].dtype == 'object' and not df[col].dropna().apply(lambda x: isinstance(x, (list, np.ndarray, dict))).any()]


for col in categorical:
    print(f"{col:30} {df[col].nunique():>6} unique values")

In [ ]:
categorical_plot = ['lang', 'doc_type', 'venue']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, col in zip(axes, categorical_plot):
    top10 = df[col].value_counts().head(10)
    ax.barh(top10.index.astype(str)[::-1], top10.values[::-1], color='steelblue', alpha=0.7)
    ax.set_title(f'Top 10 {col}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Count', fontsize=11)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

Looking at the three plots, language is heavily skewed towards English, which is expected for a scientific dataset. Document types are fairly balanced between journal and conference papers, with book-chapters being a small minority. For venue, CoRR dominates.CoRR is the Computing Research Repository, a preprint server hosted on arXiv where researchers share papers before formal publication. It is followed by IEEE Access and arXiv itself, though their counts are relatively small compared to the total dataset size, reflecting the high diversity of publication venues.

Looking at the venue plot, we can already spot a normalization issue: "IEEE Access" and "IEEE ACCESS" are clearly the same venue written in different cases, and the same applies to "Sensors" and "SENSORS". This confirms the importance of the normalization step we apply before feature extraction, where we lowercase and clean all text fields to avoid treating the same entity as multiple different ones.

### 4.3. First Similarity Check

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
import numpy as np

text = (df["title"].fillna("") + " " + df["abstract"].fillna("")).fillna("")

vectorizer = TfidfVectorizer(max_features=2000, stop_words="english")
X = vectorizer.fit_transform(text)
coords = PCA(n_components=2).fit_transform(X.toarray())

plot_df = df[["title", "year", "venue", "doc_type", "lang"]].copy()
plot_df["x"] = coords[:, 0]
plot_df["y"] = coords[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, col in zip(axes, ["doc_type", "lang"]):
    categories = plot_df[col].fillna("unknown").unique()
    colors = plt.cm.tab10(np.linspace(0, 1, len(categories)))
    for cat, color in zip(categories, colors):
        mask = plot_df[col].fillna("unknown") == cat
        ax.scatter(plot_df.loc[mask, "x"], plot_df.loc[mask, "y"],
                   label=cat, color=color, s=2, alpha=0.4)
    ax.set_title(f"PCA by {col}", fontsize=13, fontweight="bold")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.legend(markerscale=4, fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

del plot_df, X, coords, text, vectorizer
gc.collect()

Let's visualize now the similarity distribution between papers. This gives us an intuition of how separable the document space is and whether textual features alone carry enough signal for our prediction task.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

text = (df["title"].fillna("") + " " + df["abstract"].fillna("")).tolist()

embeddings = model.encode(
    text,
    batch_size=512,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

np.save("data/embeddings_visualization.npy", embeddings)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
emb = torch.tensor(embeddings, device="cuda")
idx = np.random.choice(len(emb), size=5000, replace=False)
emb_sample = emb[idx]

sims = (emb_sample @ emb_sample.T).cpu().numpy()

In [ ]:
upper = sims[np.triu_indices(len(emb_sample), k=1)]

plt.figure(figsize=(12, 5))
plt.hist(upper, bins=100, color="steelblue", alpha=0.7, edgecolor="none")
plt.axvline(upper.mean(), color="red", linestyle="--", linewidth=2, label=f"Mean: {upper.mean():.3f}")
plt.axvline(np.median(upper), color="green", linestyle="--", linewidth=2, label=f"Median: {np.median(upper):.3f}")
plt.title("Distribution of pairwise cosine similarity between papers")
plt.xlabel("Cosine similarity")
plt.ylabel("Count")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
del emb, emb_sample, sims, upper
torch.cuda.empty_cache()
gc.collect()

## 5. Dataset Split Train/Val/Test

In [ ]:
# Variabili in memoria e loro dimensione
for var_name in dir():
    var = eval(var_name)
    try:
        size_mb = len(str(var)) / 1024 / 1024
        if size_mb > 10:  # mostra solo variabili > 10 MB
            print(f"{var_name}: {size_mb:.2f} MB")
    except:
        pass

In [ ]:
# legge tutto il file in un df servono circa 30gb ram
SPLITS_EXIST = all(
    os.path.exists(f"data/{name}.parquet") 
    for name in ["train", "val", "test"]
)

if not SPLITS_EXIST:
    df_clean = pd.read_parquet("data/dataset.parquet")
    print("df_clean loaded")
else:
    print("Splits already exist, skipping dataset load.")

In [ ]:
def split_timeseries_with_gaps(df, year_col, train_size=0.7, val_size=0.15):
    years = sorted(df[year_col].unique())
    n_total_rows = len(df)
    
    # Calcoliamo le righe cumulate per anno
    rows_per_year = df.groupby(year_col).size()
    cumulative_rows = rows_per_year.cumsum()
    
    train_threshold = n_total_rows * train_size
    val_threshold   = n_total_rows * (train_size + val_size)
    
    # Troviamo l'anno in cui superiamo le soglie
    train_end_year = cumulative_rows[cumulative_rows <= train_threshold].index.max()
    val_end_year   = cumulative_rows[cumulative_rows <= val_threshold].index.max()
    
    # Gap: escludiamo 1 anno dopo ogni split
    train_end_idx = years.index(train_end_year)
    val_start_idx = train_end_idx + 2   # +1 gap, +1 per lo start
    
    val_end_idx   = years.index(val_end_year)
    test_start_idx = val_end_idx + 2
    
    years_train = years[:train_end_idx + 1]
    years_val   = years[val_start_idx : val_end_idx + 1]
    years_test  = years[test_start_idx:]
    
    train = df[df[year_col].isin(years_train)].copy()
    val   = df[df[year_col].isin(years_val)].copy()
    test  = df[df[year_col].isin(years_test)].copy()
    
    # Report
    print(f"Train: {len(train):>7} righe ({len(train)/n_total_rows:.1%}) | anni {years_train[0]}–{years_train[-1]}")
    print(f"Gap:   {years[train_end_idx+1]}")
    print(f"Val:   {len(val):>7} righe ({len(val)/n_total_rows:.1%})   | anni {years_val[0]}–{years_val[-1]}")
    print(f"Gap:   {years[val_end_idx+1]}")
    print(f"Test:  {len(test):>7} righe ({len(test)/n_total_rows:.1%})  | anni {years_test[0]}–{years_test[-1]}")
    
    return train, val, test, (years_train, years_val, years_test)

if not SPLITS_EXIST:
    df_train, df_val, df_test, periods = split_timeseries_with_gaps(df_clean, 'year')

    print(f"Train years: {periods[0]}")
    print(f"Val years:   {periods[1]}")
    print(f"Test years:  {periods[2]}")

    del df_clean
    gc.collect()
else:
    print("Splits already exist, skipping.")
gc.collect()

In [ ]:
if not SPLITS_EXIST:
    splits = {"train": df_train, "val": df_val, "test": df_test}

    for name, df in splits.items():
        file_path = os.path.join("data", f"{name}.parquet")
        df.to_parquet(file_path, engine='pyarrow', index=False)
        print(f"Saved: {file_path} | Rows: {len(df)}")
else:
    print("Files already saved, skipping.")

## Data Quality and Normalization

qui dobbiamo normalizzare testo, DOI, venue, doc_type e nomi autore, creare flag di qualità dati, evidenziare inconsistenze utili per il task finale

In [ ]:
import re
import math
import unicodedata
from collections import defaultdict

CURRENT_YEAR = 2026
MIN_REASONABLE_YEAR = 1950

ZERO_WIDTH_PATTERN = re.compile(r"[\u200b-\u200d\ufeff]")
MULTISPACE_PATTERN = re.compile(r"\s+")
PUNCT_TO_SPACE_PATTERN = re.compile(r"[^a-z0-9]+")
DOI_PREFIX_PATTERN = re.compile(r"^(https?://(dx\.)?doi\.org/|doi:)", re.IGNORECASE)

VENUE_REPLACEMENTS = {
    "intl": "international",
    "int'l": "international",
    "conf": "conference",
    "proc": "proceedings",
    "symp": "symposium",
    "sympos": "symposium",
    "worksh": "workshop",
    "trans": "transactions",
    "j": "journal",
}

DOC_TYPE_RULES = [
    ("conference", ("conference", "proceedings", "symposium", "workshop")),
    ("journal", ("journal", "transactions", "magazine")),
    ("book-chapter", ("chapter", "book chapter")),
    ("book", ("book",)),
    ("preprint", ("preprint", "posted content")),
    ("dissertation", ("dissertation", "thesis")),
    ("report", ("report", "technical report")),
]

def is_empty(x):
    if x is None:
        return True
    if isinstance(x, float) and math.isnan(x):
        return True
    if isinstance(x, str):
        return x.strip() == ""
    if isinstance(x, np.ndarray):
        return x.size == 0
    if isinstance(x, (list, tuple, set, dict)):
        return len(x) == 0
    return False

def ensure_list(x):
    if x is None:
        return []
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, list):
        return x
    if isinstance(x, tuple):
        return list(x)
    return []

def clean_string(x):
    if x is None:
        return ""
    if isinstance(x, float) and math.isnan(x):
        return ""
    return str(x).strip()

def normalize_text(x, lowercase=True):
    text = clean_string(x)
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = ZERO_WIDTH_PATTERN.sub("", text)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text.lower() if lowercase else text

def strip_accents(x):
    text = normalize_text(x, lowercase=True)
    if not text:
        return ""
    decomposed = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in decomposed if not unicodedata.combining(ch))

def normalize_for_key(x):
    text = strip_accents(x)
    text = PUNCT_TO_SPACE_PATTERN.sub(" ", text)
    return MULTISPACE_PATTERN.sub(" ", text).strip()

def normalize_doi(x):
    doi = normalize_text(x, lowercase=True)
    if not doi:
        return ""
    doi = DOI_PREFIX_PATTERN.sub("", doi)
    return doi.strip().rstrip("/")

def normalize_author_name(x):
    name = normalize_text(x, lowercase=False)
    if not name:
        return ""

    if "," in name:
        left, right = [part.strip() for part in name.split(",", 1)]
        if left and right:
            name = f"{right} {left}"

    name = normalize_text(name, lowercase=True)
    name = re.sub(r"[.'`]", "", name)
    name = re.sub(r"[^a-z0-9\s\-]", " ", strip_accents(name))
    return MULTISPACE_PATTERN.sub(" ", name).strip()

def normalize_venue(x):
    venue = normalize_text(x, lowercase=True)
    if not venue:
        return ""

    venue = venue.replace("&", " and ")
    venue = re.sub(r"[./,;:()\-]+", " ", venue)

    tokens = []
    for token in venue.split():
        token = token.strip(".")
        tokens.append(VENUE_REPLACEMENTS.get(token, token))

    venue = " ".join(tokens)
    venue = strip_accents(venue)
    venue = re.sub(r"\bproc of the\b", "proceedings of", venue)
    venue = re.sub(r"\bproc\b", "proceedings", venue)
    return MULTISPACE_PATTERN.sub(" ", venue).strip()

def normalize_doc_type(x):
    text = normalize_text(x, lowercase=True)
    if not text:
        return ""
    for canonical, cues in DOC_TYPE_RULES:
        if any(cue in text for cue in cues):
            return canonical
    return text

def normalize_lang(x):
    lang = normalize_text(x, lowercase=True)
    return lang[:5] if lang else ""

def contains_non_ascii(text):
    return any(ord(ch) > 127 for ch in text)


In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

for split in ["train", "val", "test"]:
    path = f"data/{split}.parquet"
    norm_path = f"data/{split}_normalized.parquet"
    
    if os.path.exists(norm_path):
        print(f"[{split}] already exists, skipping")
        continue

    chunk_size = 200_000
    writer = None
    
    parquet_file = pq.ParquetFile(path)
    total_rows = parquet_file.metadata.num_rows
    processed = 0
    
    for batch in parquet_file.iter_batches(batch_size=chunk_size):
        chunk = batch.to_pandas()
        
        chunk["title_clean"] = chunk["title"].apply(lambda x: normalize_text(x, lowercase=False))
        chunk["title_norm"] = chunk["title"].apply(normalize_for_key)
        chunk["abstract_clean"] = chunk["abstract"].apply(lambda x: normalize_text(x, lowercase=False))
        chunk["abstract_norm"] = chunk["abstract"].apply(normalize_for_key)
        chunk["venue_clean"] = chunk["venue"].apply(lambda x: normalize_text(x, lowercase=False))
        chunk["venue_norm"] = chunk["venue"].apply(normalize_venue)
        chunk["doc_type_clean"] = chunk["doc_type"].apply(lambda x: normalize_text(x, lowercase=False))
        chunk["doc_type_norm"] = chunk["doc_type"].apply(normalize_doc_type)
        chunk["doi_norm"] = chunk["doi"].apply(normalize_doi)
        chunk["lang_norm"] = chunk["lang"].apply(normalize_lang)
        chunk["year_clean"] = pd.to_numeric(chunk["year"], errors="coerce")
        chunk["keywords_norm"] = chunk["keywords"].apply(
            lambda xs: list(dict.fromkeys([normalize_for_key(x) for x in ensure_list(xs) if normalize_for_key(x)]))
        )
        chunk["n_keywords"] = chunk["keywords_norm"].apply(len)
        chunk["n_references"] = chunk["references"].apply(lambda x: len(ensure_list(x)))
        
        table = pa.Table.from_pandas(chunk, preserve_index=False)
        
        if writer is None:
            writer = pq.ParquetWriter(norm_path, table.schema, compression="snappy")
        
        writer.write_table(table)
        processed += len(chunk)
        print(f"  [{split}] {processed:,} / {total_rows:,}")
        del chunk, table
    
    if writer:
        writer.close()
    
    # Atomic swap — only replace original after successful write
    #import os
    #os.replace(norm_path, path)
    print(f"[{split}] saved — {processed:,} rows")

In [ ]:
cols_preview = [
    "title", "title_norm",
    "venue", "venue_norm",
    "doc_type", "doc_type_norm",
    "doi", "doi_norm",
    "lang", "lang_norm"
]
df_preview = pd.read_parquet("data/train.parquet", columns=cols_preview)
display(df_preview.head(10))
del df_preview

In [ ]:
def extract_author_features(authors):
    authors = ensure_list(authors)

    author_names_raw = []
    author_names_norm = []
    author_names_ascii = []
    author_ids_clean = []
    author_orgs_clean = []
    author_missing_id_count = 0
    author_missing_org_count = 0
    author_non_ascii_count = 0

    pairs = []

    for author in authors:
        if not isinstance(author, dict):
            continue

        raw_name = normalize_text(author.get("name"), lowercase=False)
        norm_name = normalize_author_name(author.get("name"))
        ascii_name = normalize_for_key(author.get("name"))
        author_id = normalize_text(author.get("id"), lowercase=False)
        author_org = normalize_text(author.get("org"), lowercase=False)

        if raw_name:
            author_names_raw.append(raw_name)
        if norm_name:
            author_names_norm.append(norm_name)
        if ascii_name:
            author_names_ascii.append(ascii_name)
        if raw_name and ascii_name:
            pairs.append((raw_name, ascii_name))

        if author_id:
            author_ids_clean.append(author_id)
        else:
            author_missing_id_count += 1

        if author_org:
            author_orgs_clean.append(author_org)
        else:
            author_missing_org_count += 1

        if raw_name and contains_non_ascii(raw_name):
            author_non_ascii_count += 1

    seen = set()
    unique_pairs = []
    for pair in pairs:
        if pair not in seen:
            seen.add(pair)
            unique_pairs.append(pair)

    raw_by_ascii = defaultdict(set)
    for raw_name, ascii_name in unique_pairs:
        raw_by_ascii[ascii_name].add(raw_name)

    flag_author_name_variants_in_paper = any(len(v) > 1 for v in raw_by_ascii.values())

    return pd.Series({
        "author_names_raw": [raw for raw, _ in unique_pairs],
        "author_names_norm": list(dict.fromkeys(author_names_norm)),
        "author_names_ascii": [ascii_name for _, ascii_name in unique_pairs],
        "author_ids_clean": list(dict.fromkeys(author_ids_clean)),
        "author_orgs_clean": list(dict.fromkeys(author_orgs_clean)),
        "n_authors": len(author_names_raw),
        "author_missing_id_count": author_missing_id_count,
        "author_missing_org_count": author_missing_org_count,
        "author_non_ascii_count": author_non_ascii_count,
        "flag_author_name_variants_in_paper": flag_author_name_variants_in_paper
    })

In [ ]:
# for split in ["train", "val", "test"]:
#     path = f"data/{split}.parquet"
#     df_clean = pd.read_parquet(path)

#     author_features = df_clean["authors"].apply(extract_author_features)
#     df_clean = pd.concat([df_clean, author_features], axis=1)

#     df_clean.to_parquet(path, index=False)
#     print(f"[{split}] saved — {len(df_clean):,} rows")
#     del df_clean, author_features
import os
import pyarrow as pa
import pyarrow.parquet as pq

for split in ["train", "val", "test"]:
    path = f"data/{split}.parquet"
    tmp_path = f"data/{split}_authors.parquet"

    chunk_size = 200_000
    writer = None

    parquet_file = pq.ParquetFile(path)
    total_rows = parquet_file.metadata.num_rows
    processed = 0

    for batch in parquet_file.iter_batches(batch_size=chunk_size):
        chunk = batch.to_pandas()

        author_features = chunk["authors"].apply(extract_author_features)
        for col in author_features.columns:
            chunk[col] = author_features[col].values
        del author_features

        table = pa.Table.from_pandas(chunk, preserve_index=False)

        if writer is None:
            writer = pq.ParquetWriter(tmp_path, table.schema, compression="snappy")

        writer.write_table(table)
        processed += len(chunk)
        print(f"  [{split}] {processed:,} / {total_rows:,}")
        del chunk, table

    if writer:
        writer.close()

    #os.replace(tmp_path, path)
    print(f"[{split}] saved — {processed:,} rows")

In [ ]:
import pyarrow.parquet as pq

cols = [
    "authors",
    "author_names_raw",
    "author_names_norm",
    "author_names_ascii",
    "n_authors",
    "author_missing_id_count",
    "author_missing_org_count",
]

parquet_file = pq.ParquetFile("data/train.parquet")
first_batch = next(parquet_file.iter_batches(batch_size=5, columns=cols))
df_preview = first_batch.to_pandas()

display(df_preview)
del df_preview

In [ ]:
def venue_doc_type_conflict(venue_norm, doc_type_norm):
    if not venue_norm or not doc_type_norm:
        return False

    venue_looks_conference = any(token in venue_norm for token in ["conference", "proceedings", "symposium", "workshop"])
    venue_looks_journal = any(token in venue_norm for token in ["journal", "transactions", "magazine"])

    return (
        (venue_looks_conference and doc_type_norm == "journal") or
        (venue_looks_journal and doc_type_norm == "conference")
    )

for split in ["train", "val", "test"]:
    path = f"data/{split}.parquet"
    df_clean = pd.read_parquet(path)

    df_clean["flag_missing_title"] = df_clean["title_clean"].eq("")
    df_clean["flag_missing_abstract"] = df_clean["abstract_clean"].eq("")
    df_clean["flag_missing_venue"] = df_clean["venue_clean"].eq("")
    df_clean["flag_missing_doc_type"] = df_clean["doc_type_clean"].eq("")
    df_clean["flag_missing_doi"] = df_clean["doi_norm"].eq("")
    df_clean["flag_missing_keywords"] = df_clean["n_keywords"].eq(0)
    df_clean["flag_missing_references"] = df_clean["n_references"].eq(0)
    df_clean["flag_missing_authors"] = df_clean["n_authors"].eq(0)
    df_clean["flag_author_id_missing_any"] = df_clean["author_missing_id_count"].gt(0)
    df_clean["flag_author_org_missing_any"] = df_clean["author_missing_org_count"].gt(0)
    df_clean["flag_author_non_ascii_present"] = df_clean["author_non_ascii_count"].gt(0)
    df_clean["flag_year_missing"] = df_clean["year_clean"].isna()
    df_clean["flag_year_suspicious"] = df_clean["year_clean"].apply(
        lambda x: False if pd.isna(x) else int(x) < MIN_REASONABLE_YEAR or int(x) > CURRENT_YEAR + 1
    )
    df_clean["flag_doc_type_unknown"] = df_clean["doc_type_norm"].eq("")
    df_clean["flag_venue_doc_type_conflict"] = [
        venue_doc_type_conflict(v, d)
        for v, d in zip(df_clean["venue_norm"], df_clean["doc_type_norm"])
    ]

    quality_cols = [
        "flag_missing_title", "flag_missing_abstract", "flag_missing_venue",
        "flag_missing_doc_type", "flag_missing_doi", "flag_missing_keywords",
        "flag_missing_references", "flag_missing_authors",
        "flag_author_id_missing_any", "flag_author_org_missing_any",
        "flag_year_missing", "flag_year_suspicious", "flag_venue_doc_type_conflict"
    ]
    df_clean["quality_score"] = df_clean[quality_cols].sum(axis=1).astype(int)

    df_clean.to_parquet(path, index=False)
    print(f"[{split}] saved — {len(df_clean):,} rows")
    del df_clean


In [ ]:
df_preview = pd.read_parquet("data/train.parquet")

quality_summary = pd.DataFrame({
    "count": df_preview[quality_cols].sum().astype(int),
    "pct": (df_preview[quality_cols].mean() * 100).round(2)
}).sort_values("count", ascending=False)
quality_summary

In [ ]:
plot_df = quality_summary.reset_index().rename(columns={"index": "flag"})

plt.figure(figsize=(12, 7))
sns.barplot(data=plot_df, x="count", y="flag", color="#D95F02")
plt.title("Data quality flags (train)")
plt.xlabel("Rows flagged")
plt.ylabel("")
plt.tight_layout()
plt.show()


In [ ]:
score_dist = (
    df_preview["quality_score"]
    .value_counts().sort_index()
    .rename_axis("quality_score").reset_index(name="count")
)
plt.figure(figsize=(10, 6))
sns.barplot(data=score_dist, x="quality_score", y="count", color="#1B9E77")
plt.title("Quality score distribution (train)")
plt.xlabel("Quality score")
plt.ylabel("Rows")
plt.tight_layout()
plt.show()

In [ ]:
year_quality = (
    df_clean.dropna(subset=["year_clean"])
    .assign(year_clean=lambda x: x["year_clean"].astype(int))
    .groupby("year_clean")[[
        "flag_missing_abstract",
        "flag_author_org_missing_any",
        "flag_venue_doc_type_conflict"
    ]]
    .mean()
    .mul(100)
    .reset_index()
)

year_quality_long = year_quality.melt(id_vars="year_clean", var_name="issue", value_name="pct")

plt.figure(figsize=(14, 6))
sns.lineplot(data=year_quality_long, x="year_clean", y="pct", hue="issue", marker="o")
plt.title("Quality issues by year (train)")
plt.xlabel("Year")
plt.ylabel("Rows flagged (%)")
plt.tight_layout()
plt.show()

del df_preview

In [ ]:
df_preview = pd.read_parquet("data/train.parquet", columns=["author_names_raw", "author_names_ascii"])

author_alias_rows = []
for _, row in df_preview.iterrows():
    for raw_name, ascii_name in zip(row["author_names_raw"], row["author_names_ascii"]):
        author_alias_rows.append({"author_ascii": ascii_name, "author_raw": raw_name})

del df_preview

author_alias_df = pd.DataFrame(author_alias_rows)
author_alias_summary = (
    author_alias_df.groupby("author_ascii")
    .agg(
        n_variants=("author_raw", "nunique"),
        examples=("author_raw", lambda x: " | ".join(sorted(set(list(x)[:10]))[:5]))
    )
    .reset_index()
    .sort_values(["n_variants", "author_ascii"], ascending=[False, True])
)
author_alias_summary = author_alias_summary[author_alias_summary["n_variants"] > 1]
display(author_alias_summary.head(20))
del author_alias_df

In [ ]:
author_alias_rows = []

for _, row in df_clean.iterrows():
    raw_names = row["author_names_raw"]
    ascii_names = row["author_names_ascii"]

    for raw_name, ascii_name in zip(raw_names, ascii_names):
        author_alias_rows.append({
            "author_ascii": ascii_name,
            "author_raw": raw_name
        })

author_alias_df = pd.DataFrame(author_alias_rows)

author_alias_summary = (
    author_alias_df.groupby("author_ascii")
    .agg(
        n_variants=("author_raw", "nunique"),
        examples=("author_raw", lambda x: " | ".join(sorted(set(list(x)[:10]))[:5]))
    )
    .reset_index()
    .sort_values(["n_variants", "author_ascii"], ascending=[False, True])
)

author_alias_summary = author_alias_summary[author_alias_summary["n_variants"] > 1]
display(author_alias_summary.head(20))


In [ ]:
top_author_alias = author_alias_summary.head(15).sort_values("n_variants", ascending=True)

plt.figure(figsize=(12, 7))
sns.barplot(data=top_author_alias, x="n_variants", y="author_ascii", color="#7570B3")
plt.title("Author aliases collapsed by normalization (train)")
plt.xlabel("Distinct raw spellings")
plt.ylabel("Author normalized key")
plt.tight_layout()
plt.show()


In [ ]:
df_preview = pd.read_parquet("data/train.parquet")

venue_alias_df = (
    df_preview.loc[df_preview["venue_norm"] != "", ["venue_clean", "venue_norm"]]
    .drop_duplicates()
)
venue_alias_summary = (
    venue_alias_df.groupby("venue_norm")
    .agg(
        n_variants=("venue_clean", "nunique"),
        examples=("venue_clean", lambda x: " | ".join(sorted(set(list(x)[:10]))[:5]))
    )
    .reset_index()
    .sort_values(["n_variants", "venue_norm"], ascending=[False, True])
)
venue_alias_summary = venue_alias_summary[venue_alias_summary["n_variants"] > 1]
display(venue_alias_summary.head(20))

In [ ]:
top_venue_alias = venue_alias_summary.head(15).sort_values("n_variants", ascending=True)

plt.figure(figsize=(12, 7))
sns.barplot(data=top_venue_alias, x="n_variants", y="venue_norm", color="#E7298A")
plt.title("Venue aliases collapsed by normalization (train)")
plt.xlabel("Distinct raw spellings")
plt.ylabel("Venue normalized key")
plt.tight_layout()
plt.show()

In [ ]:
conflict_examples = df_preview.loc[
    df_preview["flag_venue_doc_type_conflict"],
    ["id", "title", "year_clean", "venue_clean", "venue_norm", "doc_type_clean", "doc_type_norm"]
].copy()
print(f"Numero di conflitti venue/doc_type: {len(conflict_examples):,}")
display(conflict_examples.head(20))



In [ ]:
author_problem_examples = df_preview.loc[
    df_preview["flag_author_id_missing_any"] | df_preview["flag_author_org_missing_any"] | df_preview["flag_author_name_variants_in_paper"],
    [
        "id", "title", "year_clean",
        "author_names_raw", "author_names_norm",
        "author_ids_clean", "author_orgs_clean",
        "author_missing_id_count", "author_missing_org_count",
        "flag_author_name_variants_in_paper"
    ]
].copy()
display(author_problem_examples.head(20))

del df_preview

In [ ]:
quality_cols = [
    "flag_missing_title", "flag_missing_abstract", "flag_missing_venue",
    "flag_missing_doc_type", "flag_missing_doi", "flag_missing_keywords",
    "flag_missing_references", "flag_missing_authors",
    "flag_author_id_missing_any", "flag_author_org_missing_any",
    "flag_year_missing", "flag_year_suspicious", "flag_venue_doc_type_conflict"
]

cols_for_next_steps = [
    "id", "title", "title_norm", "abstract", "abstract_norm",
    "keywords", "keywords_norm", "year", "year_clean",
    "authors", "author_names_raw", "author_names_norm", "author_names_ascii",
    "author_ids_clean", "author_orgs_clean", "n_authors",
    "references", "n_references", "lang", "lang_norm",
    "doi", "doi_norm", "venue", "venue_clean", "venue_norm",
    "doc_type", "doc_type_clean", "doc_type_norm",
    "n_keywords", "quality_score"
] + quality_cols

for split in ["train", "val", "test"]:
    df = pd.read_parquet(f"data/{split}.parquet")
    df_out = df[cols_for_next_steps].copy()
    del df

    out_path = f"data/{split}_normalized.parquet"
    df_out.to_parquet(out_path, index=False)
    print(f"[{split}] {df_out.shape} → {out_path}")
    del df_out

## Graph

In [ ]:
TRAIN_PATH = "data/train_normalized.parquet"
CENTRALITY_OUTPUT = "data/train_centrality.parquet"
GRAPH_STATS_OUTPUT = "data/graph_stats.parquet"

In [ ]:
# Load only id and references columns to save RAM
t0 = time.time()
df_refs = pd.read_parquet(TRAIN_PATH, columns=["id", "references"])
print(f"Loaded {len(df_refs):,} train papers in {time.time()-t0:.1f}s")

valid_ids = set(df_refs["id"].values)
print(f"Valid IDs: {len(valid_ids):,}")

In [ ]:
# Build edge list from references (only edges where both papers are in train)
t0 = time.time()
edges = []

for _, row in df_refs.iterrows():
    source = row["id"]
    refs = row["references"]

    if not isinstance(refs, (list, np.ndarray)) or len(refs) == 0:
        continue

    for ref in refs:
        ref_str = str(ref).strip()
        if ref_str in valid_ids and ref_str != source:
            edges.append((source, ref_str))

print(f"Edge list built in {time.time()-t0:.1f}s")
print(f"Total edges: {len(edges):,}")

# Free df_refs, we only need edges now
del df_refs

# Create edge DataFrame
edges_df = pd.DataFrame(edges, columns=["src", "dst"])
del edges
print(f"Edge DataFrame shape: {edges_df.shape}")

In [ ]:
# cuGraph needs integer node IDs, so we create a mapping
all_nodes = list(valid_ids)
node_to_int = {node: i for i, node in enumerate(all_nodes)}
int_to_node = {i: node for node, i in node_to_int.items()}

# Map string IDs to integers
edges_df["src_int"] = edges_df["src"].map(node_to_int)
edges_df["dst_int"] = edges_df["dst"].map(node_to_int)

# Drop string columns to save memory
edges_int = edges_df[["src_int", "dst_int"]].rename(
    columns={"src_int": "src", "dst_int": "dst"}
)
del edges_df

print(f"Nodes: {len(all_nodes):,}")
print(f"Edges: {len(edges_int):,}")

In [ ]:
t0 = time.time()

# Convert to cuDF DataFrame (moves data to GPU)
edges_cudf = cudf.DataFrame(edges_int)
del edges_int

# Create directed graph
G = cugraph.Graph(directed=True)
G.from_cudf_edgelist(edges_cudf, source="src", destination="dst")

print(f"cuGraph graph built in {time.time()-t0:.1f}s")
print(f"  Nodes: {G.number_of_vertices():,}")
print(f"  Edges: {G.number_of_edges():,}")
print(f"  Density: {G.number_of_edges() / (G.number_of_vertices() * (G.number_of_vertices()-1)):.8f}")

del edges_cudf


In [ ]:
t0 = time.time()

# Weakly connected components (cuGraph needs undirected for WCC)
G_undirected = G.to_undirected()
wcc_df = cugraph.weakly_connected_components(G_undirected)
del G_undirected

# Analyze components
comp_sizes = wcc_df.groupby("labels").size().reset_index(name="size")
comp_sizes = comp_sizes.sort_values("size", ascending=False).reset_index(drop=True)

# Convert to pandas for analysis
comp_sizes_pd = comp_sizes.to_pandas() if hasattr(comp_sizes, 'to_pandas') else comp_sizes

n_components = len(comp_sizes_pd)
largest = comp_sizes_pd.iloc[0]["size"]
total_nodes = comp_sizes_pd["size"].sum()

print(f"Connected components analysis ({time.time()-t0:.1f}s):")
print(f"  Total components: {n_components:,}")
print(f"  Largest component: {largest:,} nodes ({largest/total_nodes*100:.1f}%)")
if n_components > 1:
    print(f"  2nd largest: {comp_sizes_pd.iloc[1]['size']:,} nodes")

# Size breakdown
isolated = (comp_sizes_pd["size"] == 1).sum()
small = ((comp_sizes_pd["size"] > 1) & (comp_sizes_pd["size"] <= 10)).sum()
medium = ((comp_sizes_pd["size"] > 10) & (comp_sizes_pd["size"] <= 100)).sum()
large = (comp_sizes_pd["size"] > 100).sum()

print(f"\n  Isolated (size=1): {isolated:,}")
print(f"  Small (2-10):      {small:,}")
print(f"  Medium (11-100):   {medium:,}")
print(f"  Large (>100):      {large:,}")

In [ ]:
#sns.set_theme(style="whitegrid", context="talk")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: histogram of component sizes (excluding largest)
small_comps = comp_sizes_pd[comp_sizes_pd["size"] < largest]["size"].values
if len(small_comps) > 0:
    axes[0].hist(small_comps, bins=50, color="#4C72B0", alpha=0.85, log=True)
    axes[0].set_title("Component sizes (excluding largest)")
    axes[0].set_xlabel("Component size")
    axes[0].set_ylabel("Count (log scale)")
else:
    axes[0].text(0.5, 0.5, "Only one component", ha="center", va="center", fontsize=14)
    axes[0].set_title("Component sizes (excluding largest)")

# Right: pie chart
rest = total_nodes - largest
axes[1].pie(
    [largest, rest],
    labels=[f"Main component\n({int(largest):,})", f"Other\n({int(rest):,})"],
    colors=["#54A24B", "#E45756"],
    autopct="%1.1f%%",
    startangle=90,
    textprops={"fontsize": 11}
)
axes[1].set_title("Main component vs rest")

plt.suptitle("Citation Graph Component Analysis", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Compute Centrality Measures

print("Computing centrality measures on GPU...\n")

# ── In-degree & Out-degree
print("In-degree & Out-degree...")
t0 = time.time()
in_degree = G.in_degree()
out_degree = G.out_degree()

# Convert to pandas
in_deg_pd = in_degree.to_pandas() if hasattr(in_degree, 'to_pandas') else in_degree
out_deg_pd = out_degree.to_pandas() if hasattr(out_degree, 'to_pandas') else out_degree

in_deg_dict = dict(zip(in_deg_pd["vertex"], in_deg_pd["degree"]))
out_deg_dict = dict(zip(out_deg_pd["vertex"], out_deg_pd["degree"]))
print(f"  Done in {time.time()-t0:.1f}s")
del in_degree, out_degree, in_deg_pd, out_deg_pd

# ── PageRank
print("PageRank...")
t0 = time.time()
pagerank_df = cugraph.pagerank(G, alpha=0.85, max_iter=100, tol=1e-06)

pr_pd = pagerank_df.to_pandas() if hasattr(pagerank_df, 'to_pandas') else pagerank_df
pr_dict = dict(zip(pr_pd["vertex"], pr_pd["pagerank"]))
print(f"  Done in {time.time()-t0:.1f}s")
del pagerank_df, pr_pd

# ── Betweenness Centrality
n_vertices = G.number_of_vertices()
k_betweenness = min(5000, n_vertices)
print(f"Betweenness centrality (k={k_betweenness:,})...")
t0 = time.time()
bc_df = cugraph.betweenness_centrality(G, k=k_betweenness, normalized=True, seed=42)

bc_pd = bc_df.to_pandas() if hasattr(bc_df, 'to_pandas') else bc_df
bc_dict = dict(zip(bc_pd["vertex"], bc_pd["betweenness_centrality"]))
print(f"  Done in {time.time()-t0:.1f}s")
del bc_df, bc_pd

print("\nAll centrality measures computed!")

# Free GPU memory
del G
gc.collect()

In [ ]:
# Build centrality for each paper (map int IDs back to string IDs)
centrality_rows = []

for node_str in all_nodes:
    node_int = node_to_int[node_str]
    centrality_rows.append({
        "id": node_str,
        "in_degree": in_deg_dict.get(node_int, 0),
        "out_degree": out_deg_dict.get(node_int, 0),
        "pagerank": pr_dict.get(node_int, 0.0),
        "betweenness": bc_dict.get(node_int, 0.0),
    })

centrality_df = pd.DataFrame(centrality_rows)
del centrality_rows, in_deg_dict, out_deg_dict, pr_dict, bc_dict

print(f"Centrality DataFrame shape: {centrality_df.shape}")
display(centrality_df.describe().round(6))

# Save
centrality_df.to_parquet(CENTRALITY_OUTPUT, index=False)
print(f"\nCentrality saved to {CENTRALITY_OUTPUT}")
print(f"Size: {Path(CENTRALITY_OUTPUT).stat().st_size / 1024**2:.1f} MB")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col, color in zip(axes.flatten(),
                           ["in_degree", "out_degree", "pagerank", "betweenness"],
                           ["#4C72B0", "#E45756", "#54A24B", "#7570B3"]):
    data = centrality_df[col]
    if col in ["in_degree", "out_degree"]:
        ax.hist(data[data > 0], bins=100, color=color, alpha=0.85, log=True)
        ax.set_ylabel("Count (log scale)")
    else:
        ax.hist(data[data > 0], bins=100, color=color, alpha=0.85)
        ax.set_ylabel("Count")
    ax.set_title(f"{col} distribution", fontsize=12)
    ax.set_xlabel(col)

plt.suptitle("Centrality Measure Distributions (Train Graph)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Free memory
#del centrality_df
#gc.collect()
#print("Memory freed. Centrality is saved on disk.")


In [ ]:
import graphistry
con = duckdb.connect()


print("1. Finding the absolute Top 1 node...")
# We changed LIMIT 5 to LIMIT 1
top_1_query = """
    SELECT node, COUNT(*) as degree
    FROM (
        SELECT src AS node FROM edges_view
        UNION ALL
        SELECT dst AS node FROM edges_view
    )
    GROUP BY node
    ORDER BY degree DESC
    LIMIT 1
"""
top_1_node = con.execute(top_1_query).df()['node'].iloc[0]
print(f"Target Node: {top_1_node}")

print("2. Extracting network tree (Hop 1 and Hop 2)...")
hop_query = f"""
    WITH Hop1 AS (
        SELECT dst AS node FROM edges_view WHERE src = '{top_1_node}'
        UNION
        SELECT src AS node FROM edges_view WHERE dst = '{top_1_node}'
    ),
    Hop2 AS (
        SELECT e.dst AS node FROM edges_view e JOIN Hop1 h ON e.src = h.node
        UNION
        SELECT e.src AS node FROM edges_view e JOIN Hop1 h ON e.dst = h.node
    )
    SELECT 'hop1' as level, node FROM Hop1 WHERE node != '{top_1_node}'
    UNION
    SELECT 'hop2' as level, node FROM Hop2 WHERE node != '{top_1_node}' AND node NOT IN (SELECT node FROM Hop1)
"""
results = con.execute(hop_query).df()

hop_1_list = results[results['level'] == 'hop1']['node'].tolist()
hop_2_list = results[results['level'] == 'hop2']['node'].tolist()

print("3. Tagging nodes for visualization...")
node_records = []
seen_nodes = set()

# Tag the 1 Target Node
node_records.append({"id": top_1_node, "type": "Target (Top 1)", "size": 30})
seen_nodes.add(top_1_node)

# Tag Hop 1 and Hop 2
for h1 in hop_1_list:
    if h1 not in seen_nodes:
        node_records.append({"id": h1, "type": "1st Degree", "size": 15})
        seen_nodes.add(h1)
        
for h2 in hop_2_list:
    if h2 not in seen_nodes:
        node_records.append({"id": h2, "type": "2nd Degree", "size": 5})
        seen_nodes.add(h2)

nodes_df = pd.DataFrame(node_records)

print("4. Fetching connecting edges...")
con.register("plot_nodes", nodes_df)
edges_query = """
    SELECT e.src, e.dst
    FROM edges_view e
    JOIN plot_nodes n1 ON e.src = n1.id
    JOIN plot_nodes n2 ON e.dst = n2.id
"""
tree_edges_df = con.execute(edges_query).df()

print("5. Plotting with Graphistry...")
# Update our color map to match the new Top 1 label
category_colors = {
    "Target (Top 1)": "red",
    "1st Degree": "blue",
    "2nd Degree": "silver"
}

g = graphistry.edges(tree_edges_df, 'src', 'dst') \
              .nodes(nodes_df, 'id') \
              .bind(
                  point_title='id',
                  point_size='size'
              ) \
              .encode_point_color(
                  'type',
                  categorical_mapping=category_colors,
                  default_mapping="black"
              ) \
              .settings(url_params={
                  'bg': '%23E5E5E5',
                  'edgeOpacity': 0.05,
                  'pointOpacity': 0.8      
              })

g.plot()

## Feature Extraction



For each pair (source_paper, target_paper) we extract features that capture
different reasons why a citation might occur. The features are divided into
3 groups, which will later be used to train separate and combined models.

**Traditional Features (9)**

Metadata-based features that don't require text analysis or network structure:

| Feature | Description |
|---------|-------------|
| `year_diff` | year(source) − year(target). Papers cite older work, so positive values indicate a valid citation direction. |
| `is_recent` | 1 if the two papers are within 3 years of each other. Captures recency bias. |
| `target_age` | 2025 − year(target). Controls for the different citation dynamics of old vs new papers. |
| `same_venue` | 1 if both papers share the same venue. Same-community papers cite each other more. |
| `same_doc_type` | 1 if both papers share the same document type (journal, conference, etc.). |
| `n_citation_target` | In-sample citation count of the target paper. Popular papers attract more citations. |
| `n_refs_source` | Number of references the source paper has. More references = higher chance of citing any given paper. |
| `n_authors_source` | Number of authors on the source paper. |
| `n_authors_target` | Number of authors on the target paper. |

**Textual Features (3)**

Content-similarity features derived from the papers' text:

| Feature | Description |
|---------|-------------|
| `abstract_sim` | Cosine similarity between TF-IDF vectors of the two abstracts. Strongest semantic signal. |
| `keyword_jaccard` | Jaccard similarity between keyword sets. Measures topical overlap via curated descriptors. |
| `title_sim` | Cosine similarity between TF-IDF vectors of the two titles. |

**Graph Features (11)**

Structural features from the citation network and co-authorship:

| Feature | Description |
|---------|-------------|
| `common_refs` | Number of shared references (bibliographic coupling). Strong co-citation signal. |
| `has_shared_author` | 1 if at least one author appears in both papers. Self-citation is common. |
| `shared_orgs` | 1 if any authors share the same organization. |
| `in_degree_source` | Incoming citations of source paper in the train graph. |
| `in_degree_target` | Incoming citations of target paper in the train graph. |
| `out_degree_source` | Outgoing references of source paper in the train graph. |
| `out_degree_target` | Outgoing references of target paper in the train graph. |
| `pagerank_source` | PageRank of source paper. Recursive importance measure. |
| `pagerank_target` | PageRank of target paper. |
| `betweenness_source` | Betweenness centrality of source paper. Bridge between communities. |
| `betweenness_target` | Betweenness centrality of target paper. |

**Pipeline Strategy**

To handle RAM constraints with millions of papers, we split extraction into two phases:
- **Phase 1 (DuckDB)**: traditional features + graph centrality lookups via SQL joins.
  DuckDB works directly on parquet files without loading everything into RAM.
- **Phase 2 (Python + GPU)**: textual similarities (TF-IDF + cosine on GPU),
  set-based features (common_refs, shared_author, shared_orgs) via Python dicts.

Centrality measures come from the **train graph only** to prevent data leakage.
Val/test papers not present in the train graph receive centrality = 0.

Each set (train, val, test) is processed independently and saved before
loading the next one, keeping peak RAM usage low.

**Total: 23 features per pair.**

In [ ]:
# Paths
DATA_DIR = Path("data")
TRAIN_NORMALIZED = DATA_DIR / "train_normalized.parquet"
VAL_NORMALIZED   = DATA_DIR / "val_normalized.parquet"
TEST_NORMALIZED  = DATA_DIR / "test_normalized.parquet"

TRAIN_PAIRS = DATA_DIR / "train_pairs.parquet"
VAL_PAIRS   = DATA_DIR / "val_pairs.parquet"
TEST_PAIRS  = DATA_DIR / "test_pairs.parquet"

CENTRALITY_PATH = DATA_DIR / "train_centrality.parquet"


# TF-IDF config
TF_IDF_MAX_FEATURES = 5000


# Phase 1 outputs
PHASE1_OUTPUTS = {
    "train": DATA_DIR / "train_features_phase1.parquet",
    "val":   DATA_DIR / "val_features_phase1.parquet",
    "test":  DATA_DIR / "test_features_phase1.parquet",
}

In [ ]:
def extract_traditional_features_duckdb(pairs_path, normalized_path, centrality_path, output_path):
    con = duckdb.connect()
    t0 = time.time()

    # Step 1: base features via joins
    tmp_path = str(output_path).replace(".parquet", "_tmp.parquet")

    con.execute(f"""
        COPY (
            SELECT
                p.source_id,
                p.target_id,
                p.label,

                -- TRADITIONAL
                COALESCE(CAST(s.year_clean AS INTEGER), 0)
                    - COALESCE(CAST(t.year_clean AS INTEGER), 0) AS year_diff,

                CASE WHEN ABS(
                    COALESCE(CAST(s.year_clean AS INTEGER), 0)
                    - COALESCE(CAST(t.year_clean AS INTEGER), 0)
                ) <= 3 THEN 1 ELSE 0 END AS is_recent,

                2025 - COALESCE(CAST(t.year_clean AS INTEGER), 2025) AS target_age,

                CASE WHEN s.venue_norm = t.venue_norm
                     AND s.venue_norm IS NOT NULL
                     AND s.venue_norm != ''
                     THEN 1 ELSE 0 END AS same_venue,

                CASE WHEN s.doc_type_norm = t.doc_type_norm
                     AND s.doc_type_norm IS NOT NULL
                     AND s.doc_type_norm != ''
                     THEN 1 ELSE 0 END AS same_doc_type,

                COALESCE(s.n_references, 0) AS n_refs_source,
                COALESCE(s.n_authors, 0)    AS n_authors_source,
                COALESCE(t.n_authors, 0)    AS n_authors_target,

                -- GRAPH CENTRALITY
                COALESCE(cs.in_degree,    0)   AS in_degree_source,
                COALESCE(ct.in_degree,    0)   AS in_degree_target,
                COALESCE(cs.out_degree,   0)   AS out_degree_source,
                COALESCE(ct.out_degree,   0)   AS out_degree_target,
                COALESCE(cs.pagerank,     0.0) AS pagerank_source,
                COALESCE(ct.pagerank,     0.0) AS pagerank_target,
                COALESCE(cs.betweenness,  0.0) AS betweenness_source,
                COALESCE(ct.betweenness,  0.0) AS betweenness_target

            FROM '{pairs_path}' p
            LEFT JOIN '{normalized_path}' s  ON p.source_id = s.id
            LEFT JOIN '{normalized_path}' t  ON p.target_id = t.id
            LEFT JOIN '{centrality_path}'  cs ON p.source_id = cs.id
            LEFT JOIN '{centrality_path}'  ct ON p.target_id = ct.id

        ) TO '{tmp_path}' (FORMAT PARQUET)
    """)

    # Step 2: compute citation counts from labelled pairs, then join & write final
    con.execute(f"""
        CREATE TABLE citation_counts AS
        SELECT target_id AS id, COUNT(*) AS n_citation_target
        FROM '{pairs_path}'
        WHERE label = 1
        GROUP BY target_id
    """)

    con.execute(f"""
        COPY (
            SELECT
                f.*,
                COALESCE(cc.n_citation_target, 0) AS n_citation_target
            FROM '{tmp_path}' f
            LEFT JOIN citation_counts cc ON f.target_id = cc.id
        ) TO '{output_path}' (FORMAT PARQUET)
    """)

    # Clean up temp file
    Path(tmp_path).unlink(missing_ok=True)

    elapsed = time.time() - t0
    n_rows = con.execute(f"SELECT COUNT(*) FROM '{output_path}'").fetchone()[0]
    con.close()

    print(f"  Rows: {n_rows:,} | Time: {elapsed:.1f}s")
    print(f"  Saved → {output_path}")

In [ ]:
split, pairs, normalized = "train", TRAIN_PAIRS, TRAIN_NORMALIZED
output = PHASE1_OUTPUTS[split]

if output.exists():
    print(f"[SKIP] {output} already exists.")
else:
    print(f"Phase 1 — {split}")
    extract_traditional_features_duckdb(pairs, normalized, CENTRALITY_PATH, output)

In [ ]:
split, pairs, normalized = "val", VAL_PAIRS, VAL_NORMALIZED
output = PHASE1_OUTPUTS[split]

if output.exists():
    print(f"[SKIP] {output} already exists.")
else:
    print(f"Phase 1 — {split}")
    extract_traditional_features_duckdb(pairs, normalized, CENTRALITY_PATH, output)

In [ ]:
split, pairs, normalized = "test", TEST_PAIRS, TEST_NORMALIZED
output = PHASE1_OUTPUTS[split]

if output.exists():
    print(f"[SKIP] {output} already exists.")
else:
    print(f"Phase 1 — {split}")
    extract_traditional_features_duckdb(pairs, normalized, CENTRALITY_PATH, output)

phase 2


In [ ]:
# Helper functions for second phase 

def jaccard(set_a, set_b):
    """Jaccard similarity between two sets."""
    if not set_a and not set_b:
        return 0.0
    intersection = len(set_a & set_b)
    union = len(set_a | set_b)
    return intersection / union if union > 0 else 0.0


def build_lookup_sets(df):
    """
    Build dictionaries for fast per-paper lookups.
    Only loads the columns we need.
    """
    ref_sets = {}
    author_sets = {}
    org_sets = {}
    keyword_sets = {}
    
    for _, row in df.iterrows():
        pid = row["id"]
        
        # References
        refs = row.get("references", [])
        if isinstance(refs, (list, np.ndarray)) and len(refs) > 0:
            ref_sets[pid] = set(str(r).strip() for r in refs)
        else:
            ref_sets[pid] = set()
        
        # Authors
        a_names = row.get("author_names_norm", [])
        author_sets[pid] = set(a_names) if isinstance(a_names, list) else set()
        
        # Organizations
        a_orgs = row.get("author_orgs_clean", [])
        org_sets[pid] = set(a_orgs) if isinstance(a_orgs, list) else set()
        
        # Keywords
        kw = row.get("keywords_norm", [])
        keyword_sets[pid] = set(kw) if isinstance(kw, list) else set()
    
    return ref_sets, author_sets, org_sets, keyword_sets


def precompute_gpu_similarities(pairs_df, id_to_idx, title_tfidf, abstract_tfidf, device):
    """
    Batch-compute cosine similarities on GPU.
    Processes in chunks to avoid VRAM overflow.
    """
    src_indices = np.array([id_to_idx[sid] for sid in pairs_df["source_id"]])
    tgt_indices = np.array([id_to_idx[tid] for tid in pairs_df["target_id"]])
    
    results = {}
    CHUNK_SIZE = 512
    
    for name, tfidf_matrix in [("title_sim", title_tfidf), ("abstract_sim", abstract_tfidf)]:
        print(f"    Computing {name} on GPU...")
        t0 = time.time()
        
        all_sims = []
        
        for chunk_start in range(0, len(src_indices), CHUNK_SIZE):
            chunk_end = min(chunk_start + CHUNK_SIZE, len(src_indices))
            
            src_idx_chunk = src_indices[chunk_start:chunk_end]
            tgt_idx_chunk = tgt_indices[chunk_start:chunk_end]
            
            src_vecs = torch.tensor(
                tfidf_matrix[src_idx_chunk].toarray(), dtype=torch.float32, device=device
            )
            tgt_vecs = torch.tensor(
                tfidf_matrix[tgt_idx_chunk].toarray(), dtype=torch.float32, device=device
            )
            
            src_norm = torch.nn.functional.normalize(src_vecs, p=2, dim=1)
            tgt_norm = torch.nn.functional.normalize(tgt_vecs, p=2, dim=1)
            
            sims = (src_norm * tgt_norm).sum(dim=1)
            all_sims.append(sims.cpu().numpy())
            
            del src_vecs, tgt_vecs, src_norm, tgt_norm, sims
            torch.cuda.empty_cache()
        
        results[name] = np.concatenate(all_sims)
        print(f"      Done in {time.time()-t0:.1f}s ({len(src_indices):,} pairs)")
    
    return results


In [ ]:
# Phase 2 — Textual + Set-based features
def extract_text_graph_features(split_name, normalized_path, phase1_path, output_path, 
                                 train_title_vectorizer=None, train_abstract_vectorizer=None):
    """
    Extract textual and set-based features for a given split.
    
    For train: fits TF-IDF vectorizers and returns them.
    For val/test: uses the train vectorizers (transform only, no fit).
    
    Produces: abstract_sim, keyword_jaccard, title_sim,
              common_refs, has_shared_author, shared_orgs
    """
    print(f"\n  Loading {split_name} data...")
    t0 = time.time()
    
    # Load only needed columns
    df = pd.read_parquet(
        normalized_path, 
        columns=["id", "title_norm", "abstract_norm", "keywords_norm", 
                 "author_names_norm", "author_orgs_clean", "references"]
    )
    print(f"    Loaded {len(df):,} papers in {time.time()-t0:.1f}s")
    
    # Load phase 1 results (has source_id, target_id, label + traditional features)
    phase1 = pd.read_parquet(phase1_path)
    print(f"    Loaded {len(phase1):,} pairs from phase 1")
    
    # Build lookup: paper_id -> row index
    id_to_idx = {pid: i for i, pid in enumerate(df["id"])}
    
    # TF-IDF 
    print(f"\n  Building TF-IDF matrices...")
    
    titles = df["title_norm"].fillna("").values
    abstracts = df["abstract_norm"].fillna("").values
    
    if train_title_vectorizer is None:
        # Train: fit + transform
        title_vectorizer = TfidfVectorizer(
            max_features=TF_IDF_MAX_FEATURES, stop_words="english",
            ngram_range=(1, 2), min_df=2
        )
        title_tfidf = title_vectorizer.fit_transform(titles)
        
        abstract_vectorizer = TfidfVectorizer(
            max_features=TF_IDF_MAX_FEATURES, stop_words="english",
            ngram_range=(1, 2), min_df=2
        )
        abstract_tfidf = abstract_vectorizer.fit_transform(abstracts)
        print("    TF-IDF: fit + transform (train)")
    else:
        # Val/Test: transform only (use train vocabulary)
        title_vectorizer = train_title_vectorizer
        abstract_vectorizer = train_abstract_vectorizer
        title_tfidf = title_vectorizer.transform(titles)
        abstract_tfidf = abstract_vectorizer.transform(abstracts)
        print("    TF-IDF: transform only (using train vocabulary)")
    
    print(f"    Title TF-IDF:    {title_tfidf.shape}")
    print(f"    Abstract TF-IDF: {abstract_tfidf.shape}")
    
    # GPU similarities
    print(f"\n  Computing GPU similarities...")
    gpu_sims = precompute_gpu_similarities(
        phase1, id_to_idx, title_tfidf, abstract_tfidf, device
    )
    
    del title_tfidf, abstract_tfidf, titles, abstracts
    gc.collect()
    torch.cuda.empty_cache()
    
    # Build lookup sets
    print(f"\n  Building lookup sets...")
    t0 = time.time()
    ref_sets, author_sets, org_sets, keyword_sets = build_lookup_sets(df)
    print(f"    Done in {time.time()-t0:.1f}s")
    
    del df
    gc.collect()
    
    # Extract set-based features 
    print(f"\n  Extracting set-based features...")
    t0 = time.time()
    
    n_pairs = len(phase1)
    keyword_jaccards = np.zeros(n_pairs)
    common_refs_arr = np.zeros(n_pairs, dtype=np.int32)
    shared_author_arr = np.zeros(n_pairs, dtype=np.int8)
    shared_orgs_arr = np.zeros(n_pairs, dtype=np.int8)
    
    for i, row in enumerate(phase1.itertuples(index=False)):
        src_id = row.source_id
        tgt_id = row.target_id
        
        # Keyword Jaccard
        keyword_jaccards[i] = jaccard(
            keyword_sets.get(src_id, set()),
            keyword_sets.get(tgt_id, set())
        )
        
        # Common references
        src_refs = ref_sets.get(src_id, set())
        tgt_refs = ref_sets.get(tgt_id, set())
        common_refs_arr[i] = len(src_refs & tgt_refs)
        
        # Shared author
        src_auth = author_sets.get(src_id, set())
        tgt_auth = author_sets.get(tgt_id, set())
        shared_author_arr[i] = int(len(src_auth & tgt_auth) > 0)
        
        # Shared org
        src_org = org_sets.get(src_id, set())
        tgt_org = org_sets.get(tgt_id, set())
        shared_orgs_arr[i] = int(len(src_org & tgt_org) > 0)
        
        if (i + 1) % 100_000 == 0:
            print(f"    [{i+1:,}/{n_pairs:,}] pairs processed")
    
    print(f"    Done in {time.time()-t0:.1f}s")
    
    del ref_sets, author_sets, org_sets, keyword_sets
    gc.collect()
    
    # Merge everything 
    print(f"\n  Merging features...")
    
    phase1["abstract_sim"] = gpu_sims["abstract_sim"]
    phase1["keyword_jaccard"] = keyword_jaccards
    phase1["title_sim"] = gpu_sims["title_sim"]
    phase1["common_refs"] = common_refs_arr
    phase1["has_shared_author"] = shared_author_arr
    phase1["shared_orgs"] = shared_orgs_arr
    
    del gpu_sims, keyword_jaccards, common_refs_arr, shared_author_arr, shared_orgs_arr
    
    # Save
    phase1.to_parquet(output_path, index=False)
    print(f"\n  Final dataset saved to {output_path}")
    print(f"  Shape: {phase1.shape}")
    print(f"  Size: {Path(output_path).stat().st_size / 1024**2:.1f} MB")
    print(f"  Columns: {list(phase1.columns)}")
    
    del phase1
    gc.collect()
    
    return title_vectorizer, abstract_vectorizer

In [ ]:
import joblib

PHASE2_OUTPUTS = {
    "train": DATA_DIR / "train_features_final.parquet",
    "val":   DATA_DIR / "val_features_final.parquet",
    "test":  DATA_DIR / "test_features_final.parquet",
}
VECTORIZER_PATH = DATA_DIR / "tfidf_vectorizers.joblib"

output = PHASE2_OUTPUTS["train"]

if output.exists() and VECTORIZER_PATH.exists():
    print(f"[SKIP] {output} already exists. Loading vectorizers...")
    title_vec, abstract_vec = joblib.load(VECTORIZER_PATH)
else:
    print("Phase 2 — train")
    title_vec, abstract_vec = extract_text_graph_features(
        "train", TRAIN_NORMALIZED,
        PHASE1_OUTPUTS["train"], output
    )
    joblib.dump((title_vec, abstract_vec), VECTORIZER_PATH)
    print(f"  Vectorizers saved → {VECTORIZER_PATH}")

In [ ]:
output = PHASE2_OUTPUTS["val"]

if output.exists():
    print(f"[SKIP] {output} already exists.")
else:
    print("Phase 2 — val")
    extract_text_graph_features(
        "val", VAL_NORMALIZED,
        PHASE1_OUTPUTS["val"], output,
        train_title_vectorizer=title_vec,
        train_abstract_vectorizer=abstract_vec
    )

In [ ]:
output = PHASE2_OUTPUTS["test"]

if output.exists():
    print(f"[SKIP] {output} already exists.")
else:
    print("Phase 2 — test")
    extract_text_graph_features(
        "test", TEST_NORMALIZED,
        PHASE1_OUTPUTS["test"], output,
        train_title_vectorizer=title_vec,
        train_abstract_vectorizer=abstract_vec
    )

del title_vec, abstract_vec
gc.collect()

In [ ]:
# sanity check 
for split in ["train", "val", "test"]:
    path = DATA_DIR / f"{split}_features_final.parquet"
    df = pd.read_parquet(path)
    print(f"\n{split}: {df.shape}")
    print(f"  Label distribution: {df['label'].value_counts().to_dict()}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Nulls: {df.isnull().sum().sum()}")
    display(df.head(3))
    del df

gc.collect()



## couple creation (duckDB)

In [ ]:
import duckdb

NEGATIVE_RATIO = 1  # cambia a 3 per provare 1:3
RANDOM_SEED = 42

def generate_pairs_duckdb(parquet_path, output_path, neg_ratio=1, seed=42):
    """
    Genera coppie positive e negative da un parquet di papers.
    Positive: paper A ha paper B nei references (entrambi nel dataset).
    Negative: coppie random dove A non cita B.
    """
    con = duckdb.connect()
    
    # Carica il parquet
    con.execute(f"CREATE TABLE papers AS SELECT * FROM '{parquet_path}'")
    
    # Tutti gli ID validi
    con.execute("CREATE TABLE valid_ids AS SELECT DISTINCT id FROM papers")
    
    # POSITIVE PAIRS 
    # Esplodi references e tieni solo quelle che puntano a paper nel dataset
    con.execute("""
        CREATE TABLE positive_pairs AS
        SELECT 
            p.id AS source_id,
            ref.ref_id AS target_id,
            1 AS label
        FROM papers p,
             LATERAL UNNEST(p.references) AS ref(ref_id)
        WHERE ref.ref_id IN (SELECT id FROM valid_ids)
          AND ref.ref_id != p.id
          AND p.references IS NOT NULL
    """)
    
    n_positive = con.execute("SELECT COUNT(*) FROM positive_pairs").fetchone()[0]
    print(f"Positive pairs: {n_positive:,}")
    
    # NEGATIVE PAIRS
    # Campiona coppie random che NON sono citazioni reali
    n_negative = n_positive * neg_ratio
    
    con.execute(f"""
        CREATE TABLE negative_pairs AS
        WITH all_ids AS (
            SELECT id FROM valid_ids
        ),
        random_pairs AS (
            SELECT 
                a.id AS source_id,
                b.id AS target_id
            FROM all_ids a, all_ids b
            WHERE a.id != b.id
            USING SAMPLE {n_negative * 3} ROWS (RESERVOIR, {seed})
        ),
        filtered AS (
            SELECT rp.source_id, rp.target_id, 0 AS label
            FROM random_pairs rp
            WHERE NOT EXISTS (
                SELECT 1 FROM positive_pairs pp
                WHERE pp.source_id = rp.source_id
                AND pp.target_id = rp.target_id
            )
            LIMIT {n_negative}
        )
        SELECT * FROM filtered
    """)
        
    n_neg_actual = con.execute("SELECT COUNT(*) FROM negative_pairs").fetchone()[0]
    print(f"Negative pairs: {n_neg_actual:,}")
    
    # COMBINA E SALVA 
    con.execute(f"""
        COPY (
            SELECT * FROM (
                SELECT * FROM positive_pairs
                UNION ALL
                SELECT * FROM negative_pairs
            )
            ORDER BY RANDOM()
        ) TO '{output_path}' (FORMAT PARQUET)
    """)
    
    total = n_positive + n_neg_actual
    print(f"Total pairs: {total:,} saved to {output_path}")
    print(f"Label distribution: 1={n_positive:,} ({n_positive/total*100:.1f}%), 0={n_neg_actual:,} ({n_neg_actual/total*100:.1f}%)")
    
    con.close()


# Genera per tutti e 3 i set 
for split in ["train", "val", "test"]:
    print(f"Generating pairs for: {split}")
    generate_pairs_duckdb(
        parquet_path=f"data/{split}_normalized.parquet",
        output_path=f"data/{split}_pairs.parquet",
        neg_ratio=NEGATIVE_RATIO,
        seed=RANDOM_SEED
    )

## Classification 


We train models using different combinations of feature groups to understand
the contribution of each group to the citation prediction task.

**Models**

We use two classifiers with different characteristics:

- **Logistic Regression** (cuML, GPU): a linear model that serves as our
  interpretable baseline. If a simple linear combination of features can
  predict citations well, we don't need complex models. Requires feature
  scaling since it's sensitive to different magnitudes.

- **LightGBM** (GPU): a gradient boosting model, generally the best
  performer on tabular data. Builds trees sequentially, each one correcting
  the errors of the previous. Handles mixed feature types (continuous and
  binary) naturally, doesn't require scaling, and is very fast on GPU.

**Feature Combinations**

We train each model on 7 feature subsets to isolate the contribution
of each group:

| Model | Features | Purpose |
|-------|----------|---------|
| M1 | Traditional | Can metadata alone predict citations? |
| M2 | Text | Can topic similarity alone predict citations? |
| M3 | Graph | Can network structure alone predict citations? |
| M4 | Traditional + Text | Does content improve metadata? |
| M5 | Traditional + Graph | Does network improve metadata? |
| M6 | Text + Graph | How do content and network combine? |
| M7 | All | What is the performance ceiling? |

**Evaluation**

We evaluate using three metrics:
- **Accuracy**: overall correctness.
- **F1 Score**: harmonic mean of precision and recall, robust to imbalance.
- **AUC-ROC**: measures ranking quality, how well the model separates
  positive from negative pairs regardless of threshold.

We train on the train set and evaluate on the validation set.
The test set is held out for final evaluation of the best model.

**Total: 2 models × 7 combinations = 14 experiments.**


In [ ]:

import numpy as np
import pandas as pd
import lightgbm as lgb
import time
import gc
import warnings
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, roc_curve
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

DATA_DIR = Path("data")

# Try to import cuML for GPU Logistic Regression, fallback to sklearn
try:
    from cuml.linear_model import LogisticRegression as cuLogisticRegression
    USE_CUML = True
    print("cuML available — Logistic Regression will run on GPU")
except ImportError:
    from sklearn.linear_model import LogisticRegression as SklearnLogisticRegression
    USE_CUML = False
    print("cuML not available — Logistic Regression will run on CPU (sklearn)")



In [ ]:
TRADITIONAL_FEATURES = [
    "year_diff",
    "is_recent",
    "target_age",
    "same_venue",
    "same_doc_type",
    "n_citation_target",
    "n_refs_source",
    "n_authors_source",
    "n_authors_target",
]

TEXT_FEATURES = [
    "abstract_sim",
    "keyword_jaccard",
    "title_sim",
]

GRAPH_FEATURES = [
    "common_refs",
    "has_shared_author",
    "shared_orgs",
    "in_degree_source",
    "in_degree_target",
    "out_degree_source",
    "out_degree_target",
    "pagerank_source",
    "pagerank_target",
    "betweenness_source",
    "betweenness_target",
]

ALL_FEATURES = TRADITIONAL_FEATURES + TEXT_FEATURES + GRAPH_FEATURES

FEATURE_COMBINATIONS = {
    "M1 Traditional":         TRADITIONAL_FEATURES,
    "M2 Text":                TEXT_FEATURES,
    "M3 Graph":               GRAPH_FEATURES,
    "M4 Traditional + Text":  TRADITIONAL_FEATURES + TEXT_FEATURES,
    "M5 Traditional + Graph": TRADITIONAL_FEATURES + GRAPH_FEATURES,
    "M6 Text + Graph":        TEXT_FEATURES + GRAPH_FEATURES,
    "M7 All":                 ALL_FEATURES,
}

print("Feature combinations:")
for name, feats in FEATURE_COMBINATIONS.items():
    print(f"  {name:30s} -> {len(feats):2d} features")


In [ ]:

print("Loading datasets...")
t0 = time.time()

train = pd.read_parquet(DATA_DIR / "train_features_final.parquet")
val = pd.read_parquet(DATA_DIR / "val_features_final.parquet")

print(f"  Train: {len(train):,} pairs")
print(f"  Val:   {len(val):,} pairs")
print(f"  Loaded in {time.time()-t0:.1f}s")

X_train = train[ALL_FEATURES].astype(np.float32)
y_train = train["label"].values
X_val = val[ALL_FEATURES].astype(np.float32)
y_val = val["label"].values

print(f"\n  Train labels: {pd.Series(y_train).value_counts().to_dict()}")
print(f"  Val labels:   {pd.Series(y_val).value_counts().to_dict()}")

del train, val
gc.collect()


In [ ]:
def get_lgbm_model():
    """LightGBM with GPU acceleration."""
    return lgb.LGBMClassifier(
        n_estimators=300,
        max_depth=12,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=50,
        subsample=0.8,
        colsample_bytree=0.8,
        device="gpu",
        random_state=42,
        verbose=-1,
        n_jobs=-1,
    )


def get_logistic_model():
    """Logistic Regression (cuML GPU or sklearn CPU)."""
    if USE_CUML:
        return cuLogisticRegression(
            max_iter=1000,
            tol=1e-4,
        )
    else:
        return SklearnLogisticRegression(
            max_iter=1000,
            tol=1e-4,
            solver="saga",
            n_jobs=-1,
            random_state=42,
        )


MODEL_FACTORIES = {
    "LightGBM": get_lgbm_model,
    "LogisticRegression": get_logistic_model,
}


In [ ]:
def evaluate_model(model, X, y):
    """Compute accuracy, F1, and AUC for a fitted model."""
    y_pred = model.predict(X)
    y_proba = np.asarray(model.predict_proba(X))[:, 1]
    
    return {
        "Accuracy": accuracy_score(y, y_pred),
        "F1": f1_score(y, y_pred),
        "AUC": roc_auc_score(y, y_proba),
    }

SCALER_PATH = DATA_DIR / "scaler.joblib"

if SCALER_PATH.exists():
    print(f"[SKIP] Loading scaler from {SCALER_PATH}")
    scaler = joblib.load(SCALER_PATH)
    X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=ALL_FEATURES)
else:
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=ALL_FEATURES)
    joblib.dump(scaler, SCALER_PATH)
    print(f"  Scaler saved → {SCALER_PATH}")

X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=ALL_FEATURES)

In [ ]:
MODELS_DIR = DATA_DIR / "models"
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_PATH = DATA_DIR / "training_results.joblib"

results = []
trained_models = {}

for model_name, model_factory in MODEL_FACTORIES.items():
    print(f"\n  {model_name}")
    
    for combo_name, feature_list in FEATURE_COMBINATIONS.items():
        model_path = MODELS_DIR / f"{model_name}_{combo_name}.joblib"

        if model_name == "LogisticRegression":
            X_tr = X_train_scaled[feature_list]
            X_va = X_val_scaled[feature_list]
        else:
            X_tr = X_train[feature_list]
            X_va = X_val[feature_list]

        if model_path.exists():
            print(f"  [SKIP] {combo_name:30s} — loading from disk")
            model = joblib.load(model_path)
            train_time = 0.0
        else:
            t0 = time.time()
            model = model_factory()
            model.fit(X_tr, y_train)
            train_time = time.time() - t0
            joblib.dump(model, model_path)

        metrics = evaluate_model(model, X_va, y_val)

        results.append({
            "Model": model_name,
            "Features": combo_name,
            "N_Features": len(feature_list),
            "Accuracy": metrics["Accuracy"],
            "F1": metrics["F1"],
            "AUC": metrics["AUC"],
            "Time (s)": round(train_time, 1),
        })

        if combo_name == "M7 All":
            trained_models[model_name] = model

        print(f"  {combo_name:30s} | Acc={metrics['Accuracy']:.4f} | F1={metrics['F1']:.4f} | AUC={metrics['AUC']:.4f}")

joblib.dump(results, RESULTS_PATH)

In [ ]:
if not results:
    results = joblib.load(RESULTS_PATH)

results_df = pd.DataFrame(results)
display(results_df.round(4))

In [ ]:
# Results Comparison

#sns.set_theme(style="whitegrid", context="talk")

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
metrics_to_plot = ["Accuracy", "F1", "AUC"]
colors = {"LightGBM": "#54A24B", "LogisticRegression": "#4C72B0"}

for ax, metric in zip(axes, metrics_to_plot):
    pivot = results_df.pivot(index="Features", columns="Model", values=metric)
    pivot = pivot.sort_values(by=pivot.columns[0], ascending=True)
    
    pivot.plot(kind="barh", ax=ax, color=[colors[c] for c in pivot.columns], alpha=0.85)
    ax.set_xlabel(metric)
    ax.set_title(metric, fontsize=13)
    ax.legend(fontsize=10)

plt.suptitle("Model Comparison: Citation Prediction", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()



In [ ]:
# Best Model per Feature Combination 
print("── Best model per feature combination (AUC) ──\n")
for combo in FEATURE_COMBINATIONS.keys():
    subset = results_df[results_df["Features"] == combo]
    best = subset.loc[subset["AUC"].idxmax()]
    print(f"  {combo:30s} -> {best['Model']:20s} (AUC={best['AUC']:.4f})")


Feature Importance — LightGBM 

In [ ]:
lgbm_full = trained_models.get("LightGBM")
if lgbm_full is not None:
    importance_df = pd.DataFrame({
        "Feature": ALL_FEATURES,
        "Importance": lgbm_full.feature_importances_
    }).sort_values("Importance", ascending=True)
    
    def get_group(feat):
        if feat in TRADITIONAL_FEATURES:
            return "Traditional"
        elif feat in TEXT_FEATURES:
            return "Text"
        else:
            return "Graph"
    
    importance_df["Group"] = importance_df["Feature"].apply(get_group)
    group_colors = {"Traditional": "#4C72B0", "Text": "#E45756", "Graph": "#54A24B"}
    
    plt.figure(figsize=(12, 9))
    plt.barh(
        importance_df["Feature"],
        importance_df["Importance"],
        color=[group_colors[g] for g in importance_df["Group"]],
        alpha=0.85
    )
    
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=c, label=g, alpha=0.85) for g, c in group_colors.items()]
    plt.legend(handles=legend_elements, loc="lower right", fontsize=11)
    
    plt.xlabel("Feature Importance (split count)")
    plt.title("Feature Importance — LightGBM Full Model (M7)", fontsize=13)
    plt.tight_layout()
    plt.show()
    
    display(importance_df.sort_values("Importance", ascending=False).round(4))


In [ ]:
# ROC Curves — Full Model (M7)
plt.figure(figsize=(10, 8))

for model_name, model in trained_models.items():
    if model_name == "LogisticRegression":
        X_eval = X_val_scaled[ALL_FEATURES]
    else:
        X_eval = X_val[ALL_FEATURES]
    
    y_proba = np.asarray(model.predict_proba(X_eval))[:, 1]
    fpr, tpr, _ = roc_curve(y_val, y_proba)
    auc_val = roc_auc_score(y_val, y_proba)
    plt.plot(fpr, tpr, label=f"{model_name} (AUC={auc_val:.3f})", linewidth=2)

plt.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random (AUC=0.500)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Full Model (M7 All Features)")
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

ROC Curves — LightGBM all combinations 

In [ ]:
plt.figure(figsize=(10, 8))

for combo_name, feature_list in FEATURE_COMBINATIONS.items():
    model = get_lgbm_model()
    model.fit(X_train[feature_list], y_train)
    y_proba = model.predict_proba(X_val[feature_list])[:, 1]
    
    fpr, tpr, _ = roc_curve(y_val, y_proba)
    auc_val = roc_auc_score(y_val, y_proba)
    plt.plot(fpr, tpr, label=f"{combo_name} (AUC={auc_val:.3f})", linewidth=2)

plt.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random (AUC=0.500)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — LightGBM All Feature Combinations")
plt.legend(loc="lower right", fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Classification Report — Best Model
best_row = results_df.loc[results_df["AUC"].idxmax()]
best_model_name = best_row["Model"]
best_combo = best_row["Features"]

print(f"Best model: {best_model_name} with {best_combo}")
print(f"AUC: {best_row['AUC']:.4f}\n")

best_model = trained_models[best_model_name]
if best_model_name == "LogisticRegression":
    X_eval = X_val_scaled[ALL_FEATURES]
else:
    X_eval = X_val[ALL_FEATURES]

y_pred = best_model.predict(X_eval)
print(classification_report(y_val, y_pred, target_names=["No Citation", "Citation"]))


In [ ]:
# Group Contribution Analysis 

best_type = best_model_name
r = results_df[results_df["Model"] == best_type].set_index("Features")

print(f"── Marginal contribution of each group ({best_type}, AUC) ──\n")

delta_text = r.loc["M4 Traditional + Text", "AUC"] - r.loc["M1 Traditional", "AUC"]
print(f"  Traditional → +Text:  {delta_text:+.4f} AUC")

delta_graph = r.loc["M5 Traditional + Graph", "AUC"] - r.loc["M1 Traditional", "AUC"]
print(f"  Traditional → +Graph: {delta_graph:+.4f} AUC")

delta_text2 = r.loc["M6 Text + Graph", "AUC"] - r.loc["M3 Graph", "AUC"]
print(f"  Graph       → +Text:  {delta_text2:+.4f} AUC")

delta_graph2 = r.loc["M6 Text + Graph", "AUC"] - r.loc["M2 Text", "AUC"]
print(f"  Text        → +Graph: {delta_graph2:+.4f} AUC")

best_duo_name = max(
    ["M4 Traditional + Text", "M5 Traditional + Graph", "M6 Text + Graph"],
    key=lambda m: r.loc[m, "AUC"]
)
delta_all = r.loc["M7 All", "AUC"] - r.loc[best_duo_name, "AUC"]
print(f"\n  Best pair ({best_duo_name}) → +All: {delta_all:+.4f} AUC")

print(f"\n── Summary ──")
best_single = max(["M1 Traditional", "M2 Text", "M3 Graph"], key=lambda m: r.loc[m, "AUC"])
print(f"  Best single group: {best_single} (AUC={r.loc[best_single, 'AUC']:.4f})")
print(f"  Best pair:         {best_duo_name} (AUC={r.loc[best_duo_name, 'AUC']:.4f})")
print(f"  Full model:        M7 All (AUC={r.loc['M7 All', 'AUC']:.4f})")



In [ ]:
# Final Evaluation on Test Set 
print("  FINAL EVALUATION ON TEST SET")

test = pd.read_parquet(DATA_DIR / "test_features_final.parquet")
X_test = test[ALL_FEATURES].astype(np.float32)
y_test = test["label"].values
print(f"\nTest set: {len(test):,} pairs")
print(f"Test labels: {pd.Series(y_test).value_counts().to_dict()}")

del test
gc.collect()

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=ALL_FEATURES
)

for model_name, model in trained_models.items():
    if model_name == "LogisticRegression":
        X_eval = X_test_scaled[ALL_FEATURES]
    else:
        X_eval = X_test[ALL_FEATURES]
    
    metrics = evaluate_model(model, X_eval, y_test)
    
    print(f"\n{'─'*50}")
    print(f"  {model_name} (M7 All)")
    print(f"  Accuracy: {metrics['Accuracy']:.4f}")
    print(f"  F1:       {metrics['F1']:.4f}")
    print(f"  AUC:      {metrics['AUC']:.4f}")
    
    y_pred = model.predict(X_eval)
    print(f"\n{classification_report(y_test, y_pred, target_names=['No Citation', 'Citation'])}")
